In [ ]:
import os,sys,json,time,hashlib,subprocess,signal,traceback
from pathlib import Path
TASK='BIOHUB_CPU_BATCH_PREP_CONTINUE_20260923_V01'
ROOT=Path('/kaggle/working/cpu_bundle')
ROOT.mkdir(parents=True,exist_ok=True)
PAYLOAD={'prepare_bundle.py': '"""One preparation process; a parent enforces the 1800 second session budget."""\nimport hashlib,importlib.metadata as md,json,os,shutil,subprocess,sys,time,traceback\nfrom pathlib import Path\nfrom urllib.parse import urlparse,unquote\nROOT=Path(\'/kaggle/working/cpu_bundle\');START=time.monotonic();DEADLINE=float(os.environ[\'CPU_PREP_DEADLINE_EPOCH\'])\ndef sha(p):return hashlib.file_digest(open(p,\'rb\'),\'sha256\').hexdigest()\ndef record(stage,**kw):\n d={\'stage\':stage,\'utc\':time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),\'elapsed_seconds\':time.monotonic()-START,**kw};(ROOT/(stage+\'.json\')).write_text(json.dumps(d,indent=2)+\'\\n\');print(\'BUNDLE\',json.dumps(d),flush=True)\ndef run(args):\n print("START_STAGE_COMMAND",args[:4],flush=True)\n p=subprocess.run(args,capture_output=True,text=True,timeout=max(1,DEADLINE-time.time()));print(p.stdout[-10000:],p.stderr[-5000:],flush=True);p.check_returncode();return p\ntry:\n assert not (ROOT/\'preparation_started.json\').exists(),\'Duplicate preparation prohibited\'\n record(\'preparation_started\',status=\'STARTED\',task=os.environ[\'CPU_BATCH_TASK_ID\'])\n (ROOT/\'base_image_packages.json\').write_text(json.dumps({d.metadata[\'Name\']:d.version for d in md.distributions()},indent=2)+\'\\n\')\n support=Path(\'/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1\')\n if not support.exists():support=Path(\'/kaggle/input/biohub-tracking-support-pack-50ep-v1\')\n assert (support/\'wheels\').is_dir()\n from packaging.utils import parse_wheel_filename\n from packaging.tags import sys_tags\n tags=set(sys_tags());available={}\n for p in sorted((support/\'wheels\').glob(\'*.whl\')):\n  name,ver,_,wtags=parse_wheel_filename(p.name)\n  if wtags & tags:available.setdefault(str(name),[]).append((str(ver),p))\n assert all(len({v for v,p in entries})==1 for entries in available.values()),\'Ambiguous support wheel versions\'\n constraints=[name+\'==\'+entries[0][0] for name,entries in available.items() if name not in [\'numpy\',\'torch\']]\n constraints+=[\'numpy==\'+md.version(\'numpy\'),\'torch==\'+md.version(\'torch\')]\n (ROOT/\'support_constraints.txt\').write_text(\'\\n\'.join(constraints)+\'\\n\')\n specs=json.loads((ROOT/\'dependency_specs.json\').read_text())\n run([sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-index\',\'--find-links\',str(support/\'wheels\'),\'-c\',str(ROOT/\'support_constraints.txt\'),\'--report\',str(ROOT/\'support_install_report.json\'),*specs])\n wheels=ROOT/\'wheels\';wheels.mkdir(exist_ok=True)\n report=json.loads((ROOT/\'support_install_report.json\').read_text())\n selected=[]\n for row in report[\'install\']:\n  src=Path(unquote(urlparse(row[\'download_info\'][\'url\']).path));assert src.is_file() and support in src.parents\n  shutil.copy2(src,wheels/src.name);selected.append(row[\'metadata\'][\'name\']+\'==\'+row[\'metadata\'][\'version\'])\n # One fixed official-PyPI supplement. Dependencies are fixed as well; no environment-wide upgrade.\n run([sys.executable,\'-m\',\'pip\',\'download\',\'--disable-pip-version-check\',\'--index-url\',\'https://pypi.org/simple\',\'--only-binary=:all:\',\'--no-deps\',\'-d\',str(wheels),\'openvino==2026.4.0\',\'openvino-telemetry==2025.2.0\'])\n selected+=[\'openvino==2026.4.0\',\'openvino-telemetry==2025.2.0\']\n (ROOT/\'requirements.lock\').write_text(\'\\n\'.join(sorted(selected))+\'\\n\')\n run([sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-index\',\'--find-links\',str(wheels),\'-r\',str(ROOT/\'requirements.lock\')])\n # Import-only full-chain dependency check; no hidden training/prediction entrypoint.\n check=run([sys.executable,\'-c\',\'import torch,numpy,zarr,polars,tracksdata,pyscipopt,geff,ilpy,blosc2,openvino; import json; print(json.dumps({k:__import__(k).__version__ for k in ["torch","numpy","zarr","polars","openvino"]}))\'])\n record(\'environment\',versions=json.loads(check.stdout.strip().splitlines()[-1]),installed_specs=selected,network_isolation=\'NOT_VERIFIED_PREPARATION_ONLINE\')\n run([sys.executable,str(ROOT/\'convert_bundle.py\')])\n manifest={p.relative_to(ROOT).as_posix():{\'bytes\':p.stat().st_size,\'sha256\':sha(p)} for p in sorted(ROOT.rglob(\'*\')) if p.is_file() and p.name!=\'bundle_manifest.json\'}\n (ROOT/\'bundle_manifest.json\').write_text(json.dumps({\'files\':manifest,\'status\':\'PREPARED_NOT_OFFLINE_VERIFIED\'},indent=2)+\'\\n\')\n record(\'preparation_complete\',status=\'PREPARED\',files=len(manifest),total_bytes=sum(v[\'bytes\'] for v in manifest.values()))\nexcept BaseException as e:\n record(\'preparation_error\',status=\'STOPPED_ERROR\',error_type=type(e).__name__,error=str(e),traceback=traceback.format_exc());raise\n', 'convert_bundle.py': '"""Bounded real-input diagnostic only. Never execute the production notebook."""\nimport ast, contextlib, hashlib, importlib.util, json, math, os, resource, subprocess, sys, time, traceback\nfrom pathlib import Path\nROOT=Path(\'/kaggle/working/cpu_bundle\'); ROOT.mkdir(exist_ok=True)\nSTART=time.monotonic(); DEADLINE=float(os.environ[\'CPU_PREP_DEADLINE_EPOCH\'])\n\ndef receipt(stage, **data):\n    data.update(stage=stage,utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()),elapsed_seconds=time.monotonic()-START,peak_rss_kib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)\n    p=ROOT/(stage+\'.json\'); tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(data,indent=2,allow_nan=False)+\'\\n\');tmp.replace(p)\n    print(\'RECEIPT\',json.dumps(data,allow_nan=False),flush=True)\n    return data\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef guard():\n    if time.time()>=DEADLINE:raise TimeoutError(\'30 minute unified preparation budget exhausted\')\n\ndef select_defs(path,names,namespace):\n    tree=ast.parse(path.read_text());nodes=[n for n in tree.body if isinstance(n,(ast.FunctionDef,ast.ClassDef)) and n.name in names]\n    assert {n.name for n in nodes}==set(names)\n    # Exact function/class AST; excludes all training and full-video entrypoints.\n    exec(compile(ast.Module(body=nodes,type_ignores=[]),str(path),\'exec\'),namespace)\n\ndef main():\n    import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, zarr, openvino as ov\n    from torch.utils.checkpoint import checkpoint as grad_ckpt\n    from collections.abc import Sequence\n    torch.set_grad_enabled(False);torch.set_default_dtype(torch.float32)\n    threads=2;torch.set_num_threads(threads);torch.set_num_interop_threads(1)\n    torch.set_float32_matmul_precision(\'highest\')\n    # Tracing-compatible eager attention path, equally used by PT reference and conversion.\n    torch.backends.mha.set_fastpath_enabled(False)\n    for name in [\'matmul\',\'conv\',\'rnn\']:\n        backend=getattr(torch.backends.mkldnn,name,None)\n        if backend is not None and hasattr(backend,\'fp32_precision\'):backend.fp32_precision=\'ieee\'\n    assert not torch.cuda.is_available(), \'CPU session required\'\n    support=Path(\'/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1\')\n    if not support.exists():support=Path(\'/kaggle/input/biohub-tracking-support-pack-50ep-v1\')\n    repo=support/\'repo\'; weights=support/\'weights/unet_transformer/split_0/edge_predictor_best.pth\'\n    expected=json.loads((ROOT/\'source_hashes.json\').read_text())\n    actual={n:sha(repo/n) for n in expected}; assert actual==expected, \'support source checksum mismatch\'\n    t=time.monotonic();wh=sha(weights);assert wh==\'12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771\'\n    ns={\'torch\':torch,\'nn\':nn,\'F\':F,\'np\':np,\'Path\':Path,\'json\':json,\'math\':math,\'Sequence\':Sequence,\'grad_ckpt\':grad_ckpt,\'_grad_ckpt\':grad_ckpt,\'_POS_EMBED_DIM\':8,\'DEFAULT_SCALE\':(1.625,.40625,.40625)}\n    select_defs(repo/\'src/biohub_tracking/models/temporal_unet.py\',[\'_conv_block\',\'_TemporalAttention\',\'TemporalUNet3D\'],ns)\n    select_defs(repo/\'src/biohub_tracking/models/simple_node_transformer.py\',[\'CrossAttentionBlock\',\'SimpleNodeTransformer\'],ns)\n    select_defs(repo/\'scripts/train_unet_transformer.py\',[\'UNetNodeTransformer\',\'extract_pos_features\'],ns)\n    ns[\'_DEFAULT_CONFIG\']={\'unet_out_channels\':32,\'unet_layers\':[32,64,128],\'downsample\':[1,4,4],\'window_size\':2}\n    select_defs(repo/\'scripts/predict_unet_transformer.py\',[\'load_model\',\'_load_frame\',\'pool_kernel_from_um\',\'_detect_cells_pooled\'],ns)\n    select_defs(repo/\'src/biohub_tracking/io.py\',[\'_parse_scale\'],ns)\n    W=2;ds=(1,4,4)\n    comp=next(p for p in [Path(\'/kaggle/input/competitions/biohub-cell-tracking-during-development\'),Path(\'/kaggle/input/biohub-cell-tracking-during-development\')] if p.exists())\n    movies=sorted((comp/\'test\').glob(\'*.zarr\'),key=lambda p:p.stem);assert movies\n    movie=movies[0];g=zarr.open_group(str(movie),mode=\'r\'); arr=g[\'0\'];attrs=dict(g.attrs);scale=ns[\'_parse_scale\'](attrs);q=attrs[\'image_statistics\'][\'quantiles\'];ql=float(q[\'0.001\']);qh=float(q[\'0.999\']);shape=list(arr.shape);target=[-(-s//d) for s,d in zip(shape[1:],ds)];assert shape[0]>=16\n    assert movie.stem==\'44b6_0113de3b\'\n    selected=[0]\n    other=[]\n    for path in movies[1:]:\n        shape2=list(zarr.open_group(str(path),mode=\'r\')[\'0\'].shape)\n        if shape2[1:]!=shape[1:]:\n            other=[{\'video\':path.stem,\'raw_shape\':shape2,\'window\':[0,1]}];break\n    manifest={\'video\':movie.stem,\'raw_shape\':shape,\'frames\':list(range(16)), \'original_last_frame\':shape[0]-1,\'segment_end_is_true_end\':False,\'coordinate_mapping\':\'identity original frame and voxel coordinates\',\'window\':W,\'downsample\':list(ds),\'batch\':1,\'selector_mode\':\'FIXED_DIAGNOSTIC_CONFIG\',\'additional_native_shape_window\':other,\'shape_generalization\':\'SELECTED_NOT_RUN\' if other else \'SHAPE_GENERALIZATION_NOT_COVERED\',\'context\':{\'minimum_track\':6,\'rescue_minimum\':4,\'window\':2,\'smoothing_radius\':2,\'segment_length\':16},\'boundary_effect\':\'truncation can affect future association, division, rescue and smoothing; original last-frame protection remains true video end\'}\n    (ROOT/\'sample_manifest.json\').write_text(json.dumps(manifest,indent=2)+\'\\n\')\n    receipt(\'02_samples_frozen\',**manifest)\n    model,W,ds=ns[\'load_model\'](weights,torch.device(\'cpu\'));model.float();assert W==2 and ds==(1,4,4)\n    receipt(\'01_model_loaded\',weight_sha256=wh,source_sha256=actual,load_hash_build_seconds=time.monotonic()-t,window=W,downsample=ds,actual_batch=1,original_cli_unet_batch_size=4,cli_batch_note=\'original predict_video encodes one window; argument unused\',torch=torch.__version__,openvino=ov.__version__,threads=torch.get_num_threads(),interop=torch.get_num_interop_threads(),cuda_available=False,parameter_dtypes=sorted({str(p.dtype) for p in model.parameters()}))\n    xs=[];t=time.monotonic()\n    for s in selected:\n        x=torch.stack([ns[\'_load_frame\'](arr,i,target,ds) for i in range(s,s+W)])\n        x=((x-ql)/(qh-ql+1e-6)).clamp(0).unsqueeze(0).float().contiguous();assert list(x.shape)==[1,W,*target];xs.append(x)\n    receipt(\'03_real_inputs_loaded\',seconds=time.monotonic()-t,shapes=[list(x.shape) for x in xs],dtypes=[str(x.dtype) for x in xs],finite=[bool(torch.isfinite(x).all()) for x in xs],input_sha256=[hashlib.sha256(x.numpy().tobytes()).hexdigest() for x in xs])\n    class Encoder(nn.Module):\n        def __init__(self,m):super().__init__();self.m=m;self.calls=0\n        def forward(self,x):\n            self.calls+=1\n            feat,det=self.m.encode(x)\n            return (feat,*det)\n    enc=Encoder(model).eval()\n    guard();t=time.monotonic();before=enc.calls\n    converted=ov.convert_model(enc,example_input=xs[0],input=list(xs[0].shape))\n    conversion_seconds=time.monotonic()-t\n    receipt(\'05_converted\',seconds=conversion_seconds,example_input_window=selected[0],tracing_forward_calls=enc.calls-before,outputs=len(converted.outputs))\n    assert len(converted.outputs)==3\n    t=time.monotonic();ov.save_model(converted,ROOT/\'encoder.xml\',compress_to_fp16=False)\n    receipt(\'06_saved\',seconds=time.monotonic()-t,compress_to_fp16=False,xml_bytes=(ROOT/\'encoder.xml\').stat().st_size,bin_bytes=(ROOT/\'encoder.bin\').stat().st_size,xml_sha256=sha(ROOT/\'encoder.xml\'),bin_sha256=sha(ROOT/\'encoder.bin\'))\n    del converted\n    core=ov.Core();t=time.monotonic();reloaded=core.read_model(ROOT/\'encoder.xml\');reload_seconds=time.monotonic()-t\n    const_types=sorted({str(n.get_output_element_type(0)) for n in reloaded.get_ops() if n.get_type_name()==\'Constant\'})\n    assert all(\'float16\' not in x and \'bfloat16\' not in x and x not in [\'f16\',\'bf16\'] for x in const_types)\n    opts={\'INFERENCE_PRECISION_HINT\':\'f32\',\'INFERENCE_NUM_THREADS\':threads,\'NUM_STREAMS\':1,\'PERFORMANCE_HINT\':\'LATENCY\'}\n    t=time.monotonic();cm=core.compile_model(reloaded,\'CPU\',opts);compile_seconds=time.monotonic()-t\n    props={k:str(cm.get_property(k)) for k in opts};assert \'f32\' in props[\'INFERENCE_PRECISION_HINT\'] or \'float32\' in props[\'INFERENCE_PRECISION_HINT\']\n    receipt(\'07_reloaded_compiled\',reload_seconds=reload_seconds,compile_seconds=compile_seconds,device=\'CPU\',requested=opts,actual_properties=props,constant_types=const_types,output_types=[str(x.get_element_type()) for x in reloaded.outputs],offline_reload=\'local IR read/compile; network isolation NOT_VERIFIED\')\n    outputs=cm([xs[0].numpy()])\n    receipt(\'08_reload_inference\',output_shapes=[list(outputs[o].shape) for o in cm.outputs],finite=[bool(np.isfinite(outputs[o]).all()) for o in cm.outputs],note=\'preparation connected session; NOT an offline proof\')\n    runtime=cm.get_runtime_model()\n    execution_precisions={}\n    for op in runtime.get_ops():\n        info=op.get_rt_info()\n        if \'runtimePrecision\' in info:\n            k=str(info[\'runtimePrecision\']);execution_precisions[k]=execution_precisions.get(k,0)+1\n    receipt(\'09_precision\',execution_precisions=execution_precisions)\n\nif __name__==\'__main__\':\n    try:main()\n    except BaseException as e:\n        receipt(\'99_error\',error=str(e),traceback=traceback.format_exc());raise\n', 'install_offline.py': '"""Install only the prepared, hash-verified local wheels; never enable networking."""\nimport hashlib,importlib.metadata as md,json,subprocess,sys,time\nfrom pathlib import Path\nroot=Path(sys.argv[1]);started=time.monotonic();manifest=json.loads((root/\'bundle_manifest.json\').read_text())\nfor name,meta in manifest[\'files\'].items():\n p=root/name;assert p.stat().st_size==meta[\'bytes\'];assert hashlib.file_digest(open(p,\'rb\'),\'sha256\').hexdigest()==meta[\'sha256\'],name\np=subprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'--disable-pip-version-check\',\'--no-index\',\'--find-links\',str(root/\'wheels\'),\'-r\',str(root/\'requirements.lock\')],capture_output=True,text=True,timeout=600)\nprint(p.stdout,p.stderr,flush=True)\nr={\'status\':\'INSTALLED\' if p.returncode==0 else \'OFFLINE_INSTALL_FAILED\',\'seconds\':time.monotonic()-started,\'bundle_files_hash_verified\':len(manifest[\'files\']),\'pip_returncode\':p.returncode,\'pip_args\':[\'--no-index\',\'--find-links\',\'mounted cpu_bundle/wheels\']}\nPath(\'/kaggle/working/offline_install_receipt.json\').write_text(json.dumps(r,indent=2)+\'\\n\');p.check_returncode()\n', 'chain.py': '"""Two genuine sequential short chains, no selector and no formal submission."""\nimport ast,csv,hashlib,importlib.util,json,os,resource,shutil,sys,time,traceback\nfrom pathlib import Path\nHERE=Path(__file__).parent;ROOT=Path(\'/kaggle/working/cpu_offline_chain\');ROOT.mkdir(exist_ok=True)\nsys.path.insert(0,str(HERE/\'source\'));sys.path.insert(0,str(HERE))\nfrom trace_runtime import Trace\n\ndef sha(p):return hashlib.file_digest(open(p,\'rb\'),\'sha256\').hexdigest()\ndef mount(owner,slug,kind=\'datasets\'):\n paths=[Path(\'/kaggle/input\')/kind/owner/slug,Path(\'/kaggle/input\')/slug]\n return next(p for p in paths if p.is_dir())\ndef main():\n import torch,numpy as np,openvino as ov,tracksdata as td\n torch.set_grad_enabled(False);torch.set_num_threads(2);torch.set_num_interop_threads(1);torch.set_float32_matmul_precision(\'highest\');torch.backends.mha.set_fastpath_enabled(False)\n for name in [\'matmul\',\'conv\',\'rnn\']:\n  backend=getattr(torch.backends.mkldnn,name,None)\n  if backend is not None and hasattr(backend,\'fp32_precision\'):backend.fp32_precision=\'ieee\'\n assert not torch.cuda.is_available()\n bundle=Path(os.environ[\'CPU_BUNDLE\']);sample=json.loads((bundle/\'sample_manifest.json\').read_text());assert sample[\'frames\']==list(range(16))\n primary=mount(\'pilkwang\',\'biohub-tracking-support-pack-50ep-v1\');secondary=mount(\'pilkwang\',\'biohub-temporal-unet3d-seed314159-v1\');dc=mount(\'pilkwang\',\'biohub-deepcenter-unet3d-center-prior-v1\');gate=mount(\'sailorren\',\'biohub-division-train-20260914\',\'kernels\')\n weights={\'primary\':primary/\'weights/unet_transformer/split_0/edge_predictor_best.pth\',\'secondary\':secondary/\'weights/unet_transformer/split_0/edge_predictor_best.pth\',\'deepcenter\':dc/\'weights/full_frame_center/best.pt\'}\n expected={\'primary\':\'12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771\',\'secondary\':\'9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f\',\'deepcenter\':\'8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0\'}\n for k,p in weights.items():assert sha(p)==expected[k],k\n for n,h in json.loads((HERE/\'source_hashes.json\').read_text()).items():assert sha(primary/\'repo\'/n)==h\n repo=Path(\'/tmp/cpu_chain_repo\');shutil.copytree(primary/\'repo\',repo)\n shutil.copy2(HERE/\'source/predict_unet_transformer.py\',repo/\'scripts/predict_unet_transformer.py\')\n sys.path.insert(0,str(repo/\'src\'));sys.path.insert(0,str(repo/\'scripts\'))\n import predict_unet_transformer as pred\n results={}\n for arm in [\'reference\',\'cpu\']:\n  tr=Trace(arm,ROOT/arm);pred.TRACE=tr;arm_start=time.monotonic()\n  g={\'__name__\':\'diagnostic_configuration\'}\n  for i in [1,2,3]:exec(compile((HERE/f\'source/config_{i}.py\').read_text(),f\'config_{i}\',\'exec\'),g)\n  os.environ.update(BIOHUB_EDGE_FEATURE_TTA=\'1\',BIOHUB_SECONDARY_EDGE_FEATURE_TTA=\'1\',BIOHUB_SECONDARY_EDGE_FEATURE_TTA_WEIGHT=\'0.75\',BIOHUB_GPU_SHARD=arm)\n  g.update(REPO_DIR=Path(\'/tmp\')/(\'chain_\'+arm),WORKING_DIR=ROOT/arm,SUBMISSION_PATH=ROOT/(\'diagnostic_reference.csv\' if arm==\'reference\' else \'diagnostic_cpu.csv\'),RUN_STATS_PATH=ROOT/arm/\'run_stats.csv\',GATE_ROOT=gate,EXACT_DC_WEIGHT=weights[\'deepcenter\'],test_stems=[sample[\'video\']],_trio_rescue_rows=[])\n  tr.event(\'configuration\',selector_mode=\'FIXED_DIAGNOSTIC_CONFIG\',config=g[\'CONFIG_DISPLAY\'],weights=expected,threads=torch.get_num_threads(),interop=torch.get_num_interop_threads(),cpu_max=Path(\'/sys/fs/cgroup/cpu.max\').read_text().strip(),memory_max=Path(\'/sys/fs/cgroup/memory.max\').read_text().strip(),affinity=len(os.sched_getaffinity(0)),cpu_model=next(x for x in Path(\'/proc/cpuinfo\').read_text().splitlines() if x.startswith(\'model name\')))\n  t=time.monotonic();model,W,ds=pred.load_model(weights[\'primary\'],torch.device(\'cpu\'));model.float();second,W2,ds2=pred.load_model(weights[\'secondary\'],torch.device(\'cpu\'));second.float();assert W==W2==2 and ds==ds2==(1,4,4)\n  tr.event(\'dual_model_loaded\',seconds=time.monotonic()-t)\n  original_encode=model.encode\n  if arm==\'cpu\':\n   core=ov.Core();t=time.monotonic();ir=core.read_model(bundle/\'encoder.xml\');cm=core.compile_model(ir,\'CPU\',{\'INFERENCE_PRECISION_HINT\':\'f32\',\'INFERENCE_NUM_THREADS\':2,\'NUM_STREAMS\':1,\'PERFORMANCE_HINT\':\'LATENCY\'})\n   properties={k:str(cm.get_property(k)) for k in [\'INFERENCE_PRECISION_HINT\',\'INFERENCE_NUM_THREADS\',\'NUM_STREAMS\',\'PERFORMANCE_HINT\']};assert \'float32\' in properties[\'INFERENCE_PRECISION_HINT\'] or \'f32\' in properties[\'INFERENCE_PRECISION_HINT\']\n   shape=list(cm.input().shape);tr.event(\'offline_IR_loaded_compiled\',seconds=time.monotonic()-t,properties=properties,input_shape=shape,constant_types=sorted({str(n.get_output_element_type(0)) for n in ir.get_ops() if n.get_type_name()==\'Constant\'}))\n   def encode(x):\n    if list(x.shape)!=shape:\n     tr.fallback+=1;tr.event(\'shape_fallback\',actual=list(x.shape),compiled=shape);return original_encode(x)\n    vals=cm([x.detach().numpy()]);outputs=[torch.from_numpy(vals[o].copy()) for o in cm.outputs];return outputs[0],outputs[1:]\n   model.encode=encode\n  model.encode=tr.wrap(\'primary_encode\',model.encode);model.predict_edges=tr.wrap(\'primary_association_forward_reverse\',model.predict_edges)\n  second.encode=tr.wrap(\'secondary_encode\',second.encode);second.predict_edges=tr.wrap(\'secondary_association\',second.predict_edges)\n  exec(compile((HERE/\'source/postprocess.py\').read_text(),\'postprocess.py\',\'exec\'),g)\n  t=time.monotonic();g[\'DEEPCENTER_VETO_DETECTOR\']=g[\'load_deepcenter_veto_detector\']();tr.event(\'deepcenter_loaded\',seconds=time.monotonic()-t)\n  dm=g[\'DEEPCENTER_VETO_DETECTOR\'][\'model\'];dm.forward=tr.wrap(\'deepcenter_forward\',dm.forward)\n  for name in [\'motion_relink_edges\',\'close_single_frame_gaps\',\'recover_strict_gap2\',\'add_safe_divisions_postlink\',\'filter_short_track_components\',\'prune_leaves\',\'linefit_smooth_output_graph\',\'deepcenter_score_point\',\'_sprint_score\']:\n   if name in g:g[name]=tr.wrap(name,g[name])\n  original_dc=g[\'deepcenter_score_point\']\n  def dc_score(*args,**kw):\n   value=original_dc(*args,**kw)\n   if value is not None:\n    key=\'dc_\'+hashlib.sha256(json.dumps([args[0],args[1],args[2]],default=str).encode()).hexdigest()[:20]\n    tr.tensor(key,np.asarray([value]));tr.event(\'deepcenter_decision\',key=key,score=value,gap_pass=value>=g[\'DEEPCENTER_GAP_THRESHOLD\'],division_pass=value>=g[\'DEEPCENTER_SAFE_DIV_THRESHOLD\'])\n   return value\n  g[\'deepcenter_score_point\']=dc_score\n  original_gate=g[\'_sprint_score\']\n  def gate_score(*args,**kw):\n   value=original_gate(*args,**kw)\n   if value is not None:\n    key=\'gate_\'+hashlib.sha256(json.dumps(args[1:],default=str).encode()).hexdigest()[:20];tr.tensor(key,np.asarray([value]),.95)\n   return value\n  g[\'_sprint_score\']=gate_score\n  t=time.monotonic();cfg=pred.PredictConfig(det_threshold=.960,use_ilp=True,ilp_edge_weight=-1.,ilp_appearance_weight=0.,ilp_disappearance_weight=2.,ilp_division_weight=1.2,threshold=.48)\n  coords,edges=pred.predict_video(model,g[\'TEST_DIR\']/(sample[\'video\']+\'.zarr\'),torch.device(\'cpu\'),cfg=cfg,window_size=W,max_frames=16,unet_batch_size=4,downsample=ds,secondary_model=second,secondary_edge_weight=.15,secondary_detection_weight=.80,secondary_link_mode=\'low_margin_consensus\',secondary_mix_temperature=1.,secondary_low_margin_max=.35)\n  g[\'predict_seconds\']=time.monotonic()-t;tr.tensor(\'final_detections\',coords);tr.event(\'dual_inference_completed\',seconds=g[\'predict_seconds\'],nodes=len(coords),edges=len(edges),calls=tr.calls)\n  t=time.monotonic();graph=pred.build_graph(coords,edges)\n  if graph.num_edges()>0:\n   solver=td.solvers.ILPSolver(edge_weight=cfg.ilp_edge_weight*td.EdgeAttr(\'edge_prob\'),appearance_weight=cfg.ilp_appearance_weight,disappearance_weight=cfg.ilp_disappearance_weight,division_weight=cfg.ilp_division_weight)\n   graph=solver.solve(graph);tr.event(\'ILP\',status=\'EXECUTED\',seconds=time.monotonic()-t)\n  else:tr.event(\'ILP\',status=\'NOT_TRIGGERED_NO_EDGES\')\n  folder=g[\'REPO_DIR\']/\'predictions\'/arm/\'unet_transformer/split_0\';folder.mkdir(parents=True);pred.save_graph(graph,folder/(sample[\'video\']+\'.geff\'))\n  t=time.monotonic();g[\'write_test_submission\'](\'FIXED_DIAGNOSTIC_CONFIG\');tr.event(\'postprocess_and_csv\',seconds=time.monotonic()-t,calls=tr.calls,gate=g[\'SPRINT_CALLS\'],rescue=g[\'_trio_rescue_rows\'])\n  from diagnostic_validate import validate\n  check=validate(g[\'SUBMISSION_PATH\'],sample);results[arm]={\'check\':check,\'seconds\':time.monotonic()-arm_start,\'calls\':tr.calls,\'shape_fallbacks\':tr.fallback};tr.event(\'arm_completed\',**results[arm])\n  for extra in sample[\'additional_native_shape_window\']:\n   import zarr\n   path=g[\'TEST_DIR\']/(extra[\'video\']+\'.zarr\');zg=zarr.open_group(str(path),mode=\'r\');arr=zg[\'0\'];target=[-(-v//d) for v,d in zip(arr.shape[1:],ds)];quant=zg.attrs[\'image_statistics\'][\'quantiles\'];lo=float(quant[\'0.001\']);hi=float(quant[\'0.999\'])\n   x=torch.stack([pred._load_frame(arr,i,target,ds) for i in [0,1]]);x=((x-lo)/(hi-lo+1e-6)).clamp(0).unsqueeze(0).float().contiguous()\n   t=time.monotonic();feat,dets=model.encode(x);tr.tensor(\'additional_native_features\',feat)\n   for i,det in enumerate(dets):tr.tensor(\'additional_native_det_\'+str(i),det)\n   tr.event(\'native_shape_window\',video=extra[\'video\'],raw_shape=list(arr.shape),input_shape=list(x.shape),seconds=time.monotonic()-t,shape_fallbacks=tr.fallback,note=\'shape-keyed PT fallback does not verify OpenVINO conditional-shape generalization\')\n  results[arm][\'shape_fallbacks\']=tr.fallback\n  del model,second,graph,dm\n results[\'final_graph_equal\']=results[\'reference\'][\'check\'][\'id_independent_sha256\']==results[\'cpu\'][\'check\'][\'id_independent_sha256\']\n results[\'status\']=\'SHORT_CHAIN_EXECUTED_CHECK_NUMERIC_EVENTS\';results[\'limitations\']=[\'full selector NOT_VALIDATED\',\'GPU equivalence NOT_VALIDATED\',\'full video and hidden test timing NOT_VALIDATED\',\'formal submission NOT_RUN\'];(ROOT/\'result.json\').write_text(json.dumps(results,indent=2)+\'\\n\')\nif __name__==\'__main__\':\n try:main()\n except BaseException as e:\n  (ROOT/\'error.json\').write_text(json.dumps({\'status\':\'STOPPED_ERROR\',\'error_type\':type(e).__name__,\'error\':str(e),\'traceback\':traceback.format_exc()},indent=2)+\'\\n\');raise\n', 'diagnostic_validate.py': '"""Structural core extracted from frozen check_actual_csv.py; diagnostic scope only."""\nimport csv,hashlib,json,re\nfrom collections import Counter\nfrom pathlib import Path\nfrom patch_support import invariant_graph\n\ndef digest(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\ndef validate(path,manifest):\n columns=[\'id\',\'dataset\',\'row_type\',\'node_id\',\'t\',\'z\',\'y\',\'x\',\'source_id\',\'target_id\'];groups={};rows=[]\n with Path(path).open(newline=\'\') as f:\n  rd=csv.DictReader(f);assert rd.fieldnames==columns\n  for count,row in enumerate(rd):\n   for k in columns:\n    if k not in [\'dataset\',\'row_type\']:assert re.fullmatch(r\'-?\\d+\',row[k]);row[k]=int(row[k])\n   assert row[\'id\']==count;rows.append(row);ns,es=groups.setdefault(row[\'dataset\'],({},[]))\n   if row[\'row_type\']==\'node\':\n    assert row[\'node_id\'] not in ns and min(row[k] for k in [\'node_id\',\'t\',\'z\',\'y\',\'x\'])>=0\n    assert row[\'source_id\']==row[\'target_id\']==-1\n    assert row[\'t\'] in manifest[\'frames\']\n    assert all(row[k]<size for k,size in zip([\'z\',\'y\',\'x\'],manifest[\'raw_shape\'][1:]))\n    ns[row[\'node_id\']]={k:row[k] for k in [\'t\',\'z\',\'y\',\'x\']}\n   else:\n    assert row[\'row_type\']==\'edge\' and all(row[k]==-1 for k in [\'node_id\',\'t\',\'z\',\'y\',\'x\'])\n    es.append({k:row[k] for k in [\'source_id\',\'target_id\']})\n assert set(groups)=={manifest[\'video\']}\n ns,es=groups[manifest[\'video\']];pairs=[(e[\'source_id\'],e[\'target_id\']) for e in es];assert len(set(pairs))==len(pairs)\n inc=Counter();out=Counter()\n for s,t in pairs:\n  assert s in ns and t in ns and ns[t][\'t\']==ns[s][\'t\']+1;inc[t]+=1;out[s]+=1\n assert ns and max(inc.values(),default=0)<=1 and max(out.values(),default=0)<=2\n # Actual official reader and round-trip graph equality, with ID/order invariant digest.\n import polars as pl,tracksdata as td\n scope={\'pl\':pl,\'td\':td};exec((Path(__file__).parent/\'source/official_reader.py\').read_text(),scope)\n frame=pl.DataFrame(rows);g=scope[\'build_graph_from_rows\'](frame.filter(pl.col(\'row_type\')==\'node\'),frame.filter(pl.col(\'row_type\')==\'edge\'))\n rn={int(r[\'node_id\']):{k:r[k] for k in [\'t\',\'z\',\'y\',\'x\']} for r in g.node_attrs().iter_rows(named=True)};rebuild=[{\'source_id\':int(r[\'source_id\']),\'target_id\':int(r[\'target_id\'])} for r in g.edge_attrs().iter_rows(named=True)]\n inv=digest(invariant_graph(ns,es));assert inv==digest(invariant_graph(rn,rebuild))\n return {\'status\':\'DIAGNOSTIC_STRUCTURE_PASS\',\'rows\':len(rows),\'nodes\':len(ns),\'edges\':len(es),\'id_independent_sha256\':inv,\'csv_sha256\':hashlib.sha256(Path(path).read_bytes()).hexdigest(),\'official_reader_roundtrip\':\'PASS\',\'scope\':\'fixed real short segment only\',\'has_effect_and_archive_dedup\':\'NOT_APPLICABLE_DIAGNOSTIC\'}\n', 'trace_runtime.py': '"""Small receipts; full reference tensors are temporary and never notebook Output."""\nimport hashlib,json,time,resource\nfrom pathlib import Path\nimport numpy as np\n\ndef compare(a,b):\n if a.shape!=b.shape:return {\'shape_reference\':list(a.shape),\'shape_candidate\':list(b.shape),\'allclose\':False,\'reason\':\'SHAPE_DIFFERENCE\'}\n mx=total=rm=rt=0.;finite=True;close=True;a=a.ravel();b=b.ravel()\n for i in range(0,a.size,1048576):\n  x=a[i:i+1048576].astype(\'float64\');y=b[i:i+1048576].astype(\'float64\');d=np.abs(x-y);r=d/np.maximum(np.abs(x),1e-12)\n  finite &= bool(np.isfinite(x).all() and np.isfinite(y).all());close &= bool(np.allclose(x,y,atol=1e-4,rtol=1e-3,equal_nan=False));mx=max(mx,float(d.max(initial=0)));total+=float(d.sum());rm=max(rm,float(r.max(initial=0)));rt+=float(r.sum())\n return {\'elements\':a.size,\'finite\':finite,\'allclose\':close,\'max_abs_error\':mx,\'mean_abs_error\':total/max(a.size,1),\'max_relative_error\':rm,\'mean_relative_error\':rt/max(a.size,1),\'relative_denominator_floor\':1e-12,\'atol\':1e-4,\'rtol\':1e-3,\'equal_nan\':False}\nclass Trace:\n def __init__(self,arm,root):\n  self.arm=arm;self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True);self.cache=Path(\'/tmp/cpu_chain_comparison\');self.cache.mkdir(exist_ok=True);self.start=time.monotonic();self.calls={};self.fallback=0\n def event(self,stage,**kw):\n  row={\'arm\':self.arm,\'stage\':stage,\'elapsed_seconds\':time.monotonic()-self.start,\'peak_rss_kib_process_lifetime\':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,**kw}\n  with (self.root/\'events.jsonl\').open(\'a\') as f:f.write(json.dumps(row,allow_nan=False,default=str)+\'\\n\')\n  print(\'CHAIN\',json.dumps(row,allow_nan=False,default=str),flush=True)\n def tensor(self,key,value,threshold=None):\n  a=value.detach().cpu().numpy() if hasattr(value,\'detach\') else np.asarray(value);f=self.cache/(key+\'.npy\');info={\'key\':key,\'shape\':list(a.shape),\'dtype\':str(a.dtype),\'finite\':bool(np.isfinite(a).all())}\n  if self.arm==\'reference\':np.save(f,a);info[\'status\']=\'REFERENCE_SAVED_TMP\'\n  elif f.exists():\n   b=np.load(f,mmap_mode=\'r\');info[\'comparison\']=compare(b,a)\n   if threshold is not None and b.shape==a.shape:info[\'threshold_changes\']=int(np.count_nonzero((b>threshold)!=(a>threshold)));info[\'threshold\']=threshold\n  else:info[\'status\']=\'UNMATCHED_CANDIDATE_CALL\'\n  self.event(\'numeric\',**info)\n def window(self,frames,features,det,secondary):\n  key=\'window_\'+str(frames[0]);self.tensor(key+\'_primary_features\',features)\n  for i,x in enumerate(det):self.tensor(key+\'_det_\'+str(i),x,float(np.log(.960/(1-.960)))) # logits decision also compared through actual coordinates\n  if secondary is not None:self.tensor(key+\'_secondary_features\',secondary)\n  self.event(\'window_completed\',frames=list(frames))\n def edges(self,frames,src,tgt,coords,probs,threshold):\n  key=\'edges_\'+str(frames[0]);self.tensor(key+\'_src_coords\',coords[src]);self.tensor(key+\'_tgt_coords\',coords[tgt]);self.tensor(key+\'_probabilities\',probs,threshold)\n def wrap(self,name,fn):\n  def wrapped(*a,**kw):\n   t=time.monotonic();result=fn(*a,**kw);sec=time.monotonic()-t;c=self.calls.setdefault(name,{\'calls\':0,\'seconds\':0.,\'input_shapes\':[]});c[\'calls\']+=1;c[\'seconds\']+=sec\n   if a and hasattr(a[0],\'shape\'):\n    shape=list(a[0].shape)\n    if shape not in c[\'input_shapes\']:c[\'input_shapes\'].append(shape)\n   return result\n  return wrapped\n', 'source_hashes.json': '{\n  "scripts/predict_unet_transformer.py": "c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9",\n  "scripts/train_unet_transformer.py": "c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea",\n  "src/biohub_tracking/models/temporal_unet.py": "d809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac",\n  "src/biohub_tracking/models/simple_node_transformer.py": "b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e",\n  "src/biohub_tracking/io.py": "efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd"\n}\n', 'dependency_specs.json': '[\n  "tracksdata",\n  "zarr>=3.0.10,<4",\n  "pyscipopt",\n  "geff>=1.1.3.1.1",\n  "geff-spec<1.2",\n  "ilpy>=0.5.1",\n  "polars>=1.36",\n  "blosc2",\n  "dask",\n  "imagecodecs",\n  "scikit-image>=0.24",\n  "pyarrow",\n  "rustworkx>=0.17.1",\n  "sqlalchemy>=2",\n  "numcodecs>=0.13,<0.16",\n  "donfig>=0.8",\n  "google-crc32c>=1.5",\n  "bidict>=0.23.1",\n  "psygnal>=0.14",\n  "rich",\n  "networkx>=3.2.1",\n  "pydantic>=2.11",\n  "pydantic-core",\n  "annotated-types",\n  "typing-extensions>=4.13",\n  "typing-inspection",\n  "markdown-it-py",\n  "pygments",\n  "click",\n  "cloudpickle",\n  "fsspec",\n  "partd",\n  "locket",\n  "toolz",\n  "pyyaml",\n  "ndindex",\n  "msgpack",\n  "numexpr",\n  "deprecated",\n  "wrapt",\n  "imageio",\n  "pillow",\n  "tifffile",\n  "lazy-loader",\n  "tqdm"\n]\n', 'source/config_1.py': '\'\'\'Biohub Lineage Forge\n\nResearch 3D lineage reconstruction with dual temporal models,\ndual edge-feature TTA, and conservative DeepCenter division gating.\n\nResearch edition.\'\'\'\n\nimport os\nBIOHUB_PRESET = \'harmonic_v3_division_wide\'\nBIOHUB_SCORE_AXIS = \'public 0.939 base + holdout-selected post-process configuration\'\n\nos.environ["BIOHUB_OUTPUT_FILTER_SHORT_TRACKS"] = "1"\nos.environ["BIOHUB_DET_THRESHOLD"] = "0.960"\nos.environ["BIOHUB_MOTION_RELINK_LEARNED_BONUS"] = \'1.0\'\nos.environ["BIOHUB_ILP_APPEARANCE_WEIGHT"] = "0.0"\nos.environ["BIOHUB_ILP_DISAPPEARANCE_WEIGHT"] = "2"\nos.environ["BIOHUB_GAP_CLOSE_MAX_GAP"] = "2"\nos.environ["BIOHUB_GAP_CLOSE_UM"] = "5.0"\nos.environ["BIOHUB_GAP_DENSITY_ADAPTIVE"] = "1"\nos.environ["BIOHUB_GAP_DENSITY_REFERENCE_UM"] = "6.5"\nos.environ["BIOHUB_GAP_DENSITY_GAIN"] = "0.040"\nos.environ["BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM"] = "0.125"\nos.environ["BIOHUB_GAP_DENSITY_NEIGHBORS"] = "3"\nos.environ["BIOHUB_OUTPUT_MIN_TRACK_LEN"] = "6"\nos.environ["BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS"] = "1"\nos.environ["BIOHUB_OUTPUT_GAP2_RECOVERY"] = "1"\nos.environ["BIOHUB_SAFE_DIV_MAX_UM"] = "9.0"  \n\n\nos.environ["BIOHUB_SAFE_DIV_SISTER_MAX_UM"] = "14.0"  \n\n\n\nos.environ["BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU"] = "0.6"  \nos.environ["BIOHUB_SAFE_DIV_DIVERGE_UM"] = "2.25"  \n\n\nos.environ["BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM"] = "10.0"\nos.environ["BIOHUB_SAFE_DIV_FRAME_FRAC_CAP"] = "0.0076"\nos.environ["BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP"] = "0.00375"\n\nos.environ["BIOHUB_ILP_DIVISION_WEIGHT"] = "1.2"     \nos.environ["BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE"] = "1"\nos.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN"] = "4"\nos.environ["BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB"] = "0.88"\nos.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM"] = "3.0"\nos.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC"] = "0.012"\nos.environ["BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS"] = "120"\nos.environ["BIOHUB_USE_DEEPCENTER_VETO"] = "1"\nos.environ["BIOHUB_REQUIRE_DEEPCENTER_VETO"] = "1"\nos.environ["BIOHUB_DEEPCENTER_EXPECTED_EPOCH"] = "2"\nos.environ["BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM"] = "8.5"\nos.environ["BIOHUB_DEEPCENTER_CHECKPOINT"] = "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt"\nos.environ["BIOHUB_DEEPCENTER_GAP_VETO"] = "1"\nos.environ["BIOHUB_DEEPCENTER_GAP_THRESHOLD"] = "0.25"\nos.environ["BIOHUB_DEEPCENTER_SAFE_DIV_VETO"] = "1"\nos.environ["BIOHUB_RUN_OUTPUT_DIAGNOSTICS"] = "0"\nos.environ["BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT"] = "0.15"\nos.environ["BIOHUB_BIDIRECTIONAL_FUSION_MODE"] = "harmonic_probability"\nos.environ["BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION"] = "0.90"\nos.environ["BIOHUB_DIAGNOSTIC_ARM"] = "harmonic_association_production"\nos.environ["BIOHUB_VALIDATOR_N_PER_TYPE"] = "4"\nos.environ["BIOHUB_PPSWEEP_SELECT_MARGIN"] = "0.001"\nos.environ["BIOHUB_PPSWEEP_MAX_ADJ_LOSS"] = "0.0005"\n\nos.environ["BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD"] = "0.20"\nos.environ["BIOHUB_DEEPCENTER_TTA"] = "1"\nprint("BIOHUB_PRESET:", BIOHUB_PRESET)\nprint("BIOHUB_SCORE_AXIS:", BIOHUB_SCORE_AXIS)\n# Two-wave candidate; set before configuration initialization.\nos.environ["BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT"] = "0.5"\nWAVE_ARM = \'B\'\nWAVE_EXPECTED_VELOCITY = 0.5\nWAVE_LEAF_THRESHOLD = 0.3\n', 'source/config_2.py': '\nimport json as _guard_json\nimport math as _guard_math\nimport os as _guard_os\n\n_EXPECTED_NUMERIC = {\n    "BIOHUB_DET_THRESHOLD": 0.96,\n    "BIOHUB_ILP_APPEARANCE_WEIGHT": 0.0,\n    "BIOHUB_ILP_DISAPPEARANCE_WEIGHT": 2,\n    "BIOHUB_GAP_CLOSE_UM": 5.0,\n    "BIOHUB_OUTPUT_MIN_TRACK_LEN": 6.0,\n    "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": 0.15,\n}\n\n_EXPECTED_TEXT = {\n    "BIOHUB_BIDIRECTIONAL_FUSION_MODE": "harmonic_probability",\n    "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION": "0.90",\n}\n\n_drift = {}\nfor _key, _want in _EXPECTED_NUMERIC.items():\n    _raw = _guard_os.environ.get(_key)\n    if _raw is None:\n        _drift[_key] = "missing"\n        continue\n    _got = float(_raw)\n    if not _guard_math.isclose(_got, _want, rel_tol=0.0, abs_tol=1e-12):\n        _drift[_key] = {"expected": _want, "actual": _got}\n\nfor _key, _want in _EXPECTED_TEXT.items():\n    _got = _guard_os.environ.get(_key)\n    if _got != _want:\n        _drift[_key] = {"expected": _want, "actual": _got}\n\nif _drift:\n    raise RuntimeError(\n        "Configuration drift detected: " + _guard_json.dumps(_drift, sort_keys=True)\n    )\n\nprint("Configuration guard: PASS")\nprint("Baseline: fixed-90 dual-seed clean pipeline (public LB 0.913)")\nprint("Single model-level change: harmonic mutual-support association fusion")\nprint("Reverse-time association weight: 0.200")\n', 'source/config_3.py': 'from __future__ import annotations\n\nimport csv\nimport importlib.util\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport zipfile\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\nfrom IPython.display import display\n\nCOMPETITION = "biohub-cell-tracking-during-development"\nCOMP_DIR_CANDIDATES = [\n    Path(f"/kaggle/input/competitions/{COMPETITION}"),\n    Path(f"/kaggle/input/{COMPETITION}"),\n]\nCOMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])\n\nTEST_DIR = COMP_DIR / "test"\n\nWORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")\nREPO_DIR = WORKING_DIR / "tracking_repo"\nSUBMISSION_PATH = WORKING_DIR / "submission.csv"\nRUN_STATS_PATH = WORKING_DIR / "run_stats.csv"\n\nMETHOD = "unet_transformer"\nWEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"\nEXPERIMENT_TAG = "selected_101_dual_seed_near_balanced_center_confirmed_synthetic_gap"\nTARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")\nPRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(\n    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",\n    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",\n))\nALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"\n\nDET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))\nUNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))\nUSE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"\nILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))\nILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))\nILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))\nILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))\n\n\nSLICE = ""\n\n\n\nALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"\nRUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"\n\n\nOUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))\nOUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"\nOUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"\nOUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"\nOUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"\nOUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"\nMOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))\nMOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))\nMOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))\nMOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))\nMOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))\n\nOUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"\nDIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))\nDIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))\nDIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"\nOUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"\nGAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))\nGAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))\nGAP_DENSITY_ADAPTIVE = os.environ.get("BIOHUB_GAP_DENSITY_ADAPTIVE", "0") != "0"\nGAP_DENSITY_REFERENCE_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_REFERENCE_UM", "6.5"))\nGAP_DENSITY_GAIN = float(os.environ.get("BIOHUB_GAP_DENSITY_GAIN", "0.040"))\nGAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get("BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM", "0.125"))\nGAP_DENSITY_NEIGHBORS = int(os.environ.get("BIOHUB_GAP_DENSITY_NEIGHBORS", "3"))\nGAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"\nGAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))\nGAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))\nGAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))\nGAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"\nGAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))\nGAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))\nGAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))\n\nOUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"\nOUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "6"))\nOUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"\nADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get("BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE", "0") != "0"\nSHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC", "0.10"))\nSHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN", "4"))\nSHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB", "0.82"))\nSHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM", "3.25"))\nSHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC", "0.018"))\nSHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get("BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS", "180"))\n\nOUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"\nOUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))\nOUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))\n\nOUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"\nGAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))\nGAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))\nGAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))\nGAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))\nGAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"\nGAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))\n\nOUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"\nSAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))\nSAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))\nSAFE_DIV_SISTER_SYMMETRY_TAU = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU", "0.0"))\nSAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))\nSAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))\nSAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))\n\n\nSAFE_DIV_DIVERGE_UM = float(os.environ.get("BIOHUB_SAFE_DIV_DIVERGE_UM", "2.25"))\nSAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE", "1") != "0"\nSAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get("BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN", "1") != "0"\n\n\nUSE_DEEPCENTER_VETO = os.environ.get("BIOHUB_USE_DEEPCENTER_VETO", "1") != "0"\nREQUIRE_DEEPCENTER_VETO = os.environ.get("BIOHUB_REQUIRE_DEEPCENTER_VETO", "1") != "0"\nDEEPCENTER_MANIFEST_DEFAULT = os.environ.get(\n    "BIOHUB_DEEPCENTER_MANIFEST_DEFAULT",\n    "/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",\n)\nDEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get(\n    "BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT",\n    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt",\n)\nDEEPCENTER_RELATIVE = os.environ.get("BIOHUB_DEEPCENTER_RELATIVE", "weights/full_frame_center/best.pt")\nDEEPCENTER_GAP_VETO = os.environ.get("BIOHUB_DEEPCENTER_GAP_VETO", "1") != "0"\nDEEPCENTER_SAFE_DIV_VETO = os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_VETO", "1") != "0"\nDEEPCENTER_GAP_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_THRESHOLD", "0.10"))\nDEEPCENTER_EXPECTED_EPOCH = int(os.environ.get("BIOHUB_DEEPCENTER_EXPECTED_EPOCH", "0"))\nDEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get("BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM", "0"))\nDEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get("BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD", "0.12"))\nDEEPCENTER_SCORE_WIN_Z = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_Z", "1"))\nDEEPCENTER_SCORE_WIN_YX = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_WIN_YX", "2"))\nDEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get("BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES", "8"))\n\nCONFIG_DISPLAY = {\n    "experiment_tag": EXPERIMENT_TAG,\n    "method": METHOD,\n    "weights": WEIGHTS_RELATIVE,\n    "target_artifact_slug": TARGET_ARTIFACT_SLUG,\n    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),\n    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,\n    "det_threshold": DET_THRESHOLD,\n    "unet_batch_size": UNET_BATCH_SIZE,\n    "use_ilp": USE_ILP,\n    "ilp_edge_weight": ILP_EDGE_WEIGHT,\n    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,\n    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,\n    "ilp_division_weight": ILP_DIVISION_WEIGHT,\n    "slice": SLICE,\n    "allow_pip_install": ALLOW_PIP_INSTALL,\n    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,\n    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,\n    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,\n    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,\n    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,\n    "output_motion_relink": OUTPUT_MOTION_RELINK,\n    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,\n    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,\n    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,\n    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,\n    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,\n    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,\n    "div_parent_max_um": DIV_PARENT_MAX_UM,\n    "div_sister_max_um": DIV_SISTER_MAX_UM,\n    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,\n    "output_gap_close": OUTPUT_GAP_CLOSE,\n    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,\n    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),\n    "gap_close_um": GAP_CLOSE_UM,\n    "gap_density_adaptive": GAP_DENSITY_ADAPTIVE,\n    "gap_density_reference_um": GAP_DENSITY_REFERENCE_UM,\n    "gap_density_gain": GAP_DENSITY_GAIN,\n    "gap_density_max_step_delta_um": GAP_DENSITY_MAX_STEP_DELTA_UM,\n    "gap_density_neighbors": GAP_DENSITY_NEIGHBORS,\n    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,\n    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,\n    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,\n    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,\n    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,\n    "gap_refine_win_z": GAP_REFINE_WIN_Z,\n    "gap_refine_win_yx": GAP_REFINE_WIN_YX,\n    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,\n    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,\n    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,\n    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,\n    "adaptive_short_track_rescue": ADAPTIVE_SHORT_TRACK_RESCUE,\n    "short_track_rescue_trigger_removed_frac": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC,\n    "short_track_rescue_min_len": SHORT_TRACK_RESCUE_MIN_LEN,\n    "short_track_rescue_min_mean_edge_prob": SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB,\n    "short_track_rescue_max_mean_edge_dist_um": SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM,\n    "short_track_rescue_max_nodes_frac": SHORT_TRACK_RESCUE_MAX_NODES_FRAC,\n    "short_track_rescue_max_nodes_abs": SHORT_TRACK_RESCUE_MAX_NODES_ABS,\n    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,\n    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,\n    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,\n    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,\n    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,\n    "gap2_max_step_um": GAP2_MAX_STEP_UM,\n    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,\n    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,\n    "gap2_require_context": GAP2_REQUIRE_CONTEXT,\n    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,\n    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,\n    "safe_div_max_um": SAFE_DIV_MAX_UM,\n    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,\n    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,\n    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,\n    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,\n    "use_deepcenter_add_only_gate": USE_DEEPCENTER_VETO,\n    "deepcenter_gap_add_gate": DEEPCENTER_GAP_VETO,\n    "deepcenter_safe_div_add_gate": DEEPCENTER_SAFE_DIV_VETO,\n    "deepcenter_gap_threshold": DEEPCENTER_GAP_THRESHOLD,\n    "deepcenter_expected_epoch": DEEPCENTER_EXPECTED_EPOCH,\n    "deepcenter_gap_confirm_min_span_um": DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM,\n    "deepcenter_safe_div_threshold": DEEPCENTER_SAFE_DIV_THRESHOLD,\n    "deepcenter_checkpoint_default": DEEPCENTER_CHECKPOINT_DEFAULT,\n}\n\nprint("Biohub learned UNet + node-transformer + ILP submission")\nprint("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())\nprint("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())\nprint(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))', 'source/official_reader.py': 'def build_graph_from_rows(\n    node_rows: pl.DataFrame,\n    edge_rows: pl.DataFrame,\n) -> td.graph.InMemoryGraph:\n    """Rebuild a tracksdata graph from one dataset\'s node and edge rows.\n\n    tracksdata assigns fresh node ids, so CSV ``node_id``s are remapped when\n    adding edges (matching is spatial, not by id).\n    """\n    graph = td.graph.InMemoryGraph()\n    for key in ("z", "y", "x"):\n        graph.add_node_attr_key(key, pl.Float64, -999999.0)\n\n    assigned = graph.bulk_add_nodes(\n        node_rows.select(\n            pl.col("t").cast(pl.Int64),\n            pl.col("z").cast(pl.Float64),\n            pl.col("y").cast(pl.Float64),\n            pl.col("x").cast(pl.Float64),\n        ).to_dicts()\n    )\n    id_map = dict(zip(node_rows["node_id"].to_list(), assigned, strict=True))\n\n    if edge_rows.height:\n        graph.bulk_add_edges(\n            [\n                {"source_id": id_map[s], "target_id": id_map[t]}\n                for s, t in zip(\n                    edge_rows["source_id"].to_list(), edge_rows["target_id"].to_list(), strict=True\n                )\n            ]\n        )\n\n    return graph\n', 'source/patch_support.py': '"""Only the two-wave postprocessing changes and small audit helpers. No inference."""\nimport math\nfrom collections import Counter\n\n\ndef scored_probability(value):\n    if value is None or isinstance(value, bool):\n        return None\n    try:\n        value = float(value)\n    except (TypeError, ValueError, OverflowError):\n        return None\n    if not math.isfinite(value):\n        return None\n    if value < 0.0 or value > 1.0:\n        value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n    return max(0.0, min(1.0, value))\n\n\ndef prune_leaves(nodes, edges, last_frame, threshold):\n    stats = dict(candidates=0, division_exempt=0, missing_score_exempt=0,\n                 final_frame_exempt=0, deleted=0, deleted_ids=[],\n                 last_frame=last_frame, threshold=threshold)\n    if threshold is None:\n        stats[\'skipped\'] = \'DISABLED\'\n        return nodes, edges, stats\n    if last_frame is None:\n        stats[\'skipped\'] = \'TRUE_LAST_FRAME_UNAVAILABLE\'\n        return nodes, edges, stats\n    out = Counter(int(e[\'source_id\']) for e in edges)\n    incoming = {}\n    for e in edges:\n        incoming.setdefault(int(e[\'target_id\']), []).append(e)\n    remove = set()\n    for node_id, node in nodes.items():\n        inc = incoming.get(node_id, [])\n        if out[node_id] or len(inc) != 1:\n            continue\n        stats[\'candidates\'] += 1\n        e = inc[0]\n        if int(node[\'t\']) >= last_frame:\n            stats[\'final_frame_exempt\'] += 1\n        elif out[int(e[\'source_id\'])] != 1:\n            stats[\'division_exempt\'] += 1\n        elif e.get(\'_wave_model_probability\') is None:\n            stats[\'missing_score_exempt\'] += 1\n        elif e[\'_wave_model_probability\'] < threshold:\n            remove.add(node_id)\n    stats[\'deleted\'] = len(remove)\n    stats[\'deleted_ids\'] = sorted(remove)[:20]\n    return ({i:n for i,n in nodes.items() if i not in remove},\n            [e for e in edges if int(e[\'source_id\']) not in remove and int(e[\'target_id\']) not in remove], stats)\n\n\ndef invariant_graph(nodes, edges):\n    """ID-independent exact rooted-forest encoding with rounded CSV coordinates."""\n    children = {i:[] for i in nodes}\n    parents = set()\n    for e in edges:\n        s,t = int(e[\'source_id\']),int(e[\'target_id\'])\n        children[s].append(t); parents.add(t)\n    # Children always occur later in time; bottom-up encoding also handles duplicate coordinates.\n    import hashlib,json\n    labels = {}\n    for i in sorted(nodes, key=lambda i:int(nodes[i][\'t\']), reverse=True):\n        n = nodes[i]\n        value = [int(n[\'t\']),*[max(0,int(round(float(n[k])))) for k in (\'z\',\'y\',\'x\')],sorted(labels[c] for c in children[i])]\n        labels[i] = hashlib.sha256(json.dumps(value,separators=(\',\',\':\')).encode()).hexdigest()\n    return sorted(labels[i] for i in nodes if i not in parents)\n', 'source/postprocess.py': 'import tracksdata as td\nimport numpy as np\nimport blosc2\nfrom scipy.optimize import linear_sum_assignment\nfrom scipy.spatial import cKDTree\n\nSUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\nCSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]\nVOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)\n\n\ndef graph_from_geff(path: Path):\n    graph = td.graph.IndexedRXGraph.from_geff(path)\n    return graph[0] if isinstance(graph, tuple) else graph\n\n\ndef edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:\n    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]\n    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]\n    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:\n    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]\n    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]\n    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef node_point(node: dict[str, object]) -> tuple[float, float, float]:\n    return (float(node["z"]), float(node["y"]), float(node["x"]))\n\n\ndef edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:\n    prob = edge.get("edge_prob")\n    prob_value = float(prob) if prob is not None else 0.0\n    return prob_value, -float(edge["distance_um"])\n\n\ndef _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:\n    return max(nodes_by_id) + 1 if nodes_by_id else 1\n\n\n\ndef read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:\n    if t in frame_cache:\n        return frame_cache[t]\n    zarr_path = TEST_DIR / f"{dataset}.zarr"\n    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())\n    shape = tuple(int(v) for v in meta["shape"])\n    dtype = np.dtype(meta["data_type"])\n    frame_shape = shape[1:]\n    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"\n    try:\n        raw = chunk_path.read_bytes()\n        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)\n        if arr.size == int(np.prod(frame_shape)):\n            frame = arr.reshape(frame_shape).copy()\n            frame_cache[t] = frame\n            return frame\n    except Exception:\n        pass\n    import zarr\n    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])\n    frame_cache[t] = frame\n    return frame\n\n\ndef refine_synthetic_midpoint(\n    dataset: str | None,\n    t: int,\n    midpoint: tuple[float, float, float],\n    frame_cache: dict[int, np.ndarray],\n    stats: dict[str, int],\n) -> tuple[float, float, float]:\n    if not GAP_REFINE_SYNTHETIC or dataset is None:\n        return midpoint\n    try:\n        frame = read_test_frame(dataset, t, frame_cache)\n        z, y, x = [int(round(v)) for v in midpoint]\n        z0 = max(0, z - GAP_REFINE_WIN_Z)\n        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)\n        y0 = max(0, y - GAP_REFINE_WIN_YX)\n        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)\n        x0 = max(0, x - GAP_REFINE_WIN_YX)\n        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)\n        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)\n        if patch.size == 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        baseline = float(np.percentile(patch, 20.0))\n        weights = np.maximum(patch - baseline, 0.0)\n        total = float(weights.sum())\n        if total <= 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\n        refined = (\n            float((weights * zz).sum() / total),\n            float((weights * yy).sum() / total),\n            float((weights * xx).sum() / total),\n        )\n        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:\n            stats["gap_refine_rejected_shift"] += 1\n            return midpoint\n        stats["gap_refined_synthetic"] += 1\n        return refined\n    except Exception:\n        stats["gap_refine_failed"] += 1\n        return midpoint\n\n\n\ndef _dc_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:\n    if factor <= 1:\n        return volume.astype(np.float32, copy=False)\n    z, y, x = volume.shape\n    y2 = (y // factor) * factor\n    x2 = (x // factor) * factor\n    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)\n    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))\n\n\ndef _dc_normalize_dynamic_range(volume: np.ndarray, cfg: object) -> np.ndarray:\n    vol = np.asarray(volume, dtype=np.float32)\n    lo = float(np.percentile(vol, float(getattr(cfg, "norm_lo_pct", 50.0))))\n    hi = float(np.percentile(vol, float(getattr(cfg, "norm_hi_pct", 99.5))))\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\n        return np.zeros_like(vol, dtype=np.float32)\n    ratio = (vol - lo) / (hi - lo)\n    return np.clip(\n        ratio,\n        float(getattr(cfg, "norm_clip_lo", -0.5)),\n        float(getattr(cfg, "norm_clip_hi", 6.0)),\n    ).astype(np.float32)\n\n\ndef _dc_manifest_weight_paths(manifest_path: Path) -> list[Path]:\n    if not manifest_path.exists():\n        return []\n    try:\n        manifest = json.loads(manifest_path.read_text())\n    except Exception as exc:\n        print("Could not read DeepCenter manifest:", manifest_path, type(exc).__name__, exc)\n        return []\n    root = manifest_path.parent\n    sections: list[dict[str, object]] = []\n    for section in [\n        manifest.get("model", {}),\n        manifest.get("models", {}).get("full_frame_center", {}) if isinstance(manifest.get("models", {}), dict) else {},\n        manifest.get("full_frame_center", {}),\n    ]:\n        if isinstance(section, dict):\n            sections.append(section)\n    candidates: list[Path] = []\n    for section in sections:\n        for key in ("weight_path", "path"):\n            rel = section.get(key)\n            if isinstance(rel, str) and rel:\n                candidates.append(root / rel)\n        for key in ("last_checkpoint", "best_checkpoint"):\n            item = section.get(key)\n            if isinstance(item, dict):\n                rel = item.get("path")\n                if isinstance(rel, str) and rel:\n                    candidates.append(root / rel)\n    for name in ("checkpoint_last.pt", "best.pt", "last.pt"):\n        candidates.append(root / "weights" / "full_frame_center" / name)\n        candidates.append(root / name)\n    candidates.append(root / DEEPCENTER_RELATIVE)\n    return candidates\n\n\ndef _dc_checkpoint_candidates() -> list[Path]:\n    return [EXACT_DC_WEIGHT]\n\n\ntry:\n    import torch\nexcept Exception as _dc_torch_error:\n    torch = None\n\n\nif torch is not None:\n    class _DCConvBlock3d(torch.nn.Module):\n        def __init__(self, in_channels: int, out_channels: int) -> None:\n            super().__init__()\n            groups = min(8, out_channels)\n            self.block = torch.nn.Sequential(\n                torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n                torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),\n                torch.nn.GroupNorm(groups, out_channels),\n                torch.nn.SiLU(inplace=True),\n            )\n\n        def forward(self, x):\n            return self.block(x)\n\n\n    class _DCDeepCenterUNet3D(torch.nn.Module):\n        def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:\n            super().__init__()\n            c = int(base_channels)\n            self.enc1 = _DCConvBlock3d(in_channels, c)\n            self.down1 = torch.nn.MaxPool3d(2, 2)\n            self.enc2 = _DCConvBlock3d(c, c * 2)\n            self.down2 = torch.nn.MaxPool3d(2, 2)\n            self.enc3 = _DCConvBlock3d(c * 2, c * 4)\n            self.down3 = torch.nn.MaxPool3d(2, 2)\n            self.bottleneck = _DCConvBlock3d(c * 4, c * 8)\n            self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)\n            self.dec3 = _DCConvBlock3d(c * 8, c * 4)\n            self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)\n            self.dec2 = _DCConvBlock3d(c * 4, c * 2)\n            self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)\n            self.dec1 = _DCConvBlock3d(c * 2, c)\n            self.head = torch.nn.Conv3d(c, 1, 1)\n\n        def forward(self, x):\n            e1 = self.enc1(x)\n            e2 = self.enc2(self.down1(e1))\n            e3 = self.enc3(self.down2(e2))\n            b = self.bottleneck(self.down3(e3))\n            d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\n            d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\n            d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\n            return self.head(d1)\nelse:\n    _DCConvBlock3d = None\n    _DCDeepCenterUNet3D = None\n\ndef load_deepcenter_veto_detector() -> dict[str, object] | None:\n    if not USE_DEEPCENTER_VETO:\n        print("DeepCenter add-only repair gate disabled by configuration.")\n        return None\n    if torch is None:\n        if REQUIRE_DEEPCENTER_VETO:\n            raise ImportError("torch is required for DeepCenter add-only repair gate")\n        print("DeepCenter add-only repair gate skipped because torch is unavailable.")\n        return None\n    from types import SimpleNamespace\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    load_errors: list[str] = []\n    for checkpoint_path in _dc_checkpoint_candidates():\n        if not checkpoint_path.exists():\n            continue\n        try:\n            print("Trying DeepCenter add-only gate checkpoint:", checkpoint_path)\n            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n            if not isinstance(checkpoint, dict) or "model_state" not in checkpoint:\n                raise ValueError("checkpoint has no model_state")\n            checkpoint_epoch = int(checkpoint.get("epoch", -1))\n            if DEEPCENTER_EXPECTED_EPOCH > 0 and checkpoint_epoch != DEEPCENTER_EXPECTED_EPOCH:\n                raise ValueError(\n                    f"expected DeepCenter epoch {DEEPCENTER_EXPECTED_EPOCH}, got {checkpoint_epoch}"\n                )\n            cfg = SimpleNamespace(**checkpoint.get("config", {}))\n            model = _DCDeepCenterUNet3D(base_channels=int(getattr(cfg, "base_channels", 24)))\n            model.load_state_dict(checkpoint["model_state"])\n            model.to(device)\n            model.eval()\n            print("Loaded DeepCenter add-only gate checkpoint:", checkpoint_path)\n            print("DeepCenter checkpoint epoch:", checkpoint.get("epoch"), "best_score:", checkpoint.get("best_score"))\n            return {\n                "model": model,\n                "cfg": cfg,\n                "device": device,\n                "path": checkpoint_path,\n                "torch": torch,\n            }\n        except Exception as exc:\n            load_errors.append(f"{checkpoint_path}: {type(exc).__name__}: {exc}")\n            print("Skipping incompatible DeepCenter checkpoint:", checkpoint_path, "|", type(exc).__name__, exc)\n    message = "No usable DeepCenter checkpoint found for add-only repair gate."\n    if REQUIRE_DEEPCENTER_VETO:\n        checked = "\\n".join(str(p) for p in _dc_checkpoint_candidates()[:80])\n        errors = "\\n".join(load_errors[-20:])\n        raise FileNotFoundError(message + "\\nChecked:\\n" + checked + ("\\nLoad errors:\\n" + errors if errors else ""))\n    print(message)\n    return None\n\n\ndef _dc_cache_trim(cache: dict[tuple[str, int], np.ndarray]) -> None:\n    limit = max(1, int(DEEPCENTER_SCORE_CACHE_MAX_FRAMES))\n    while len(cache) > limit:\n        cache.pop(next(iter(cache)))\n\n\ndef deepcenter_heatmap_for_frame(\n    dataset: str,\n    t: int,\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> np.ndarray | None:\n    if detector_bundle is None:\n        return None\n    key = (dataset, int(t))\n    cached = heatmap_cache.get(key)\n    if cached is not None:\n        return cached\n    model = detector_bundle["model"]\n    cfg = detector_bundle["cfg"]\n    device = detector_bundle["device"]\n    torch_mod = detector_bundle["torch"]\n    pool_factor = int(getattr(cfg, "pool_factor", 4))\n    volume = read_test_frame(dataset, int(t), frame_cache)\n    pooled = _dc_pool_frame_xy(volume, pool_factor)\n    image = _dc_normalize_dynamic_range(pooled, cfg)\n    with torch_mod.no_grad():\n        tensor = torch_mod.from_numpy(image[None, None, ...]).to(device=device, dtype=torch_mod.float32)\n        logits = model(tensor)\n        \n        \n        \n        \n        \n        if os.environ.get("BIOHUB_DEEPCENTER_TTA", "0") != "0":\n            acc = logits.clone(); nv = 1\n            for dims in [(-1,), (-2,), (-2, -1)]:\n                acc = acc + model(tensor.flip(dims)).flip(dims); nv += 1\n            if tensor.shape[-1] == tensor.shape[-2]:\n                for k in (1, 3):\n                    acc = acc + torch_mod.rot90(model(torch_mod.rot90(tensor, k, dims=(-2, -1))), -k, dims=(-2, -1)); nv += 1\n                acc = acc + model(tensor.transpose(-1, -2)).transpose(-1, -2); nv += 1\n                at = torch_mod.rot90(tensor, 1, dims=(-2, -1)).transpose(-1, -2)\n                acc = acc + torch_mod.rot90(model(at).transpose(-1, -2), -1, dims=(-2, -1)); nv += 1\n            delta = float((acc / nv - logits).abs().mean())\n            if delta == 0.0:\n                raise RuntimeError("DEEPCENTER_TTA_NO_OP: averaged veto logits identical to the single view")\n            if not getattr(deepcenter_heatmap_for_frame, "_tta_announced", False):\n                print("DEEPCENTER_TTA_ACTIVE views=", nv, "mean_abs_logit_delta=", round(delta, 6), flush=True)\n                deepcenter_heatmap_for_frame._tta_announced = True\n            logits = acc / nv\n        heatmap = torch_mod.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32, copy=False)\n    heatmap_cache[key] = heatmap\n    _dc_cache_trim(heatmap_cache)\n    return heatmap\n\n\ndef deepcenter_score_point(\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n) -> float | None:\n    if not USE_DEEPCENTER_VETO or detector_bundle is None or dataset is None:\n        return None\n    heatmap = deepcenter_heatmap_for_frame(dataset, int(t), detector_bundle, frame_cache, heatmap_cache)\n    if heatmap is None or heatmap.size == 0:\n        return None\n    cfg = detector_bundle["cfg"]\n    pool_factor = int(getattr(cfg, "pool_factor", 4))\n    z = int(round(float(point[0])))\n    y = int(round(float(point[1]) / max(pool_factor, 1)))\n    x = int(round(float(point[2]) / max(pool_factor, 1)))\n    z0, z1 = max(0, z - DEEPCENTER_SCORE_WIN_Z), min(heatmap.shape[0], z + DEEPCENTER_SCORE_WIN_Z + 1)\n    y0, y1 = max(0, y - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[1], y + DEEPCENTER_SCORE_WIN_YX + 1)\n    x0, x1 = max(0, x - DEEPCENTER_SCORE_WIN_YX), min(heatmap.shape[2], x + DEEPCENTER_SCORE_WIN_YX + 1)\n    patch = heatmap[z0:z1, y0:y1, x0:x1]\n    if patch.size == 0:\n        return None\n    score = float(np.max(patch))\n    return score if np.isfinite(score) else None\n\n\ndef deepcenter_accept_repair_point(\n    dataset: str | None,\n    t: int,\n    point: tuple[float, float, float],\n    detector_bundle: dict[str, object] | None,\n    frame_cache: dict[int, np.ndarray],\n    heatmap_cache: dict[tuple[str, int], np.ndarray],\n    stats: dict[str, int],\n    prefix: str,\n    threshold: float,\n) -> bool:\n    if not USE_DEEPCENTER_VETO:\n        return True\n    if detector_bundle is None or dataset is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    stats[f"deepcenter_{prefix}_checked"] += 1\n    score = deepcenter_score_point(dataset, int(t), point, detector_bundle, frame_cache, heatmap_cache)\n    if score is None:\n        stats[f"deepcenter_{prefix}_missing"] += 1\n        return True\n    if score < float(threshold):\n        stats[f"deepcenter_{prefix}_rejected"] += 1\n        return False\n    stats[f"deepcenter_{prefix}_accepted"] += 1\n    return True\n\ndef _position_um(node: dict[str, object]) -> np.ndarray:\n    return np.array(\n        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],\n        dtype=np.float64,\n    )\n\n\n"""Only the two-wave postprocessing changes and small audit helpers. No inference."""\nimport math\nfrom collections import Counter\n\n\ndef scored_probability(value):\n    if value is None or isinstance(value, bool):\n        return None\n    try:\n        value = float(value)\n    except (TypeError, ValueError, OverflowError):\n        return None\n    if not math.isfinite(value):\n        return None\n    if value < 0.0 or value > 1.0:\n        value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n    return max(0.0, min(1.0, value))\n\n\ndef prune_leaves(nodes, edges, last_frame, threshold):\n    stats = dict(candidates=0, division_exempt=0, missing_score_exempt=0,\n                 final_frame_exempt=0, deleted=0, deleted_ids=[],\n                 last_frame=last_frame, threshold=threshold)\n    if threshold is None:\n        stats[\'skipped\'] = \'DISABLED\'\n        return nodes, edges, stats\n    if last_frame is None:\n        stats[\'skipped\'] = \'TRUE_LAST_FRAME_UNAVAILABLE\'\n        return nodes, edges, stats\n    out = Counter(int(e[\'source_id\']) for e in edges)\n    incoming = {}\n    for e in edges:\n        incoming.setdefault(int(e[\'target_id\']), []).append(e)\n    remove = set()\n    for node_id, node in nodes.items():\n        inc = incoming.get(node_id, [])\n        if out[node_id] or len(inc) != 1:\n            continue\n        stats[\'candidates\'] += 1\n        e = inc[0]\n        if int(node[\'t\']) >= last_frame:\n            stats[\'final_frame_exempt\'] += 1\n        elif out[int(e[\'source_id\'])] != 1:\n            stats[\'division_exempt\'] += 1\n        elif e.get(\'_wave_model_probability\') is None:\n            stats[\'missing_score_exempt\'] += 1\n        elif e[\'_wave_model_probability\'] < threshold:\n            remove.add(node_id)\n    stats[\'deleted\'] = len(remove)\n    stats[\'deleted_ids\'] = sorted(remove)[:20]\n    return ({i:n for i,n in nodes.items() if i not in remove},\n            [e for e in edges if int(e[\'source_id\']) not in remove and int(e[\'target_id\']) not in remove], stats)\n\n\ndef invariant_graph(nodes, edges):\n    """ID-independent exact rooted-forest encoding with rounded CSV coordinates."""\n    children = {i:[] for i in nodes}\n    parents = set()\n    for e in edges:\n        s,t = int(e[\'source_id\']),int(e[\'target_id\'])\n        children[s].append(t); parents.add(t)\n    # Children always occur later in time; bottom-up encoding also handles duplicate coordinates.\n    import hashlib,json\n    labels = {}\n    for i in sorted(nodes, key=lambda i:int(nodes[i][\'t\']), reverse=True):\n        n = nodes[i]\n        value = [int(n[\'t\']),*[max(0,int(round(float(n[k])))) for k in (\'z\',\'y\',\'x\')],sorted(labels[c] for c in children[i])]\n        labels[i] = hashlib.sha256(json.dumps(value,separators=(\',\',\':\')).encode()).hexdigest()\n    return sorted(labels[i] for i in nodes if i not in parents)\n\n\n_wave_last_frames = {}\ndef wave_last_frame(dataset):\n    key = str(Path(TEST_DIR).resolve()), dataset\n    if key not in _wave_last_frames:\n        try:\n            # Original read_test_frame uses this exact input and TZYX time indexing.\n            path = TEST_DIR / f"{dataset}.zarr" / "0" / "zarr.json"\n            meta = json.loads(path.read_text())\n            shape = meta[\'shape\']\n            assert len(shape) == 4 and int(shape[0]) > 0\n            dims = meta.get(\'dimension_names\')\n            assert dims is None or dims[0] in (\'t\',\'time\')\n            _wave_last_frames[key] = int(shape[0]) - 1\n        except (OSError, ValueError, KeyError, TypeError, AssertionError):\n            _wave_last_frames[key] = None\n    return _wave_last_frames[key]\n\ndef motion_relink_edges(\n    nodes_by_id: dict[int, dict[str, object]],\n    stats: dict[str, int],\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_MOTION_RELINK or not nodes_by_id:\n        return []\n\n    learned_edge_probs = learned_edge_probs or {}\n\n    def learned_prob(source_id: int, target_id: int) -> float:\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\n        try:\n            value = float(value)\n        except (TypeError, ValueError):\n            return 0.0\n        if not np.isfinite(value):\n            return 0.0\n        if value < 0.0 or value > 1.0:\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n        return float(np.clip(value, 0.0, 1.0))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    for ids in ids_by_t.values():\n        ids.sort()\n\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\n    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:\n        stats["motion_relink_skipped_large_frame"] = 1\n        return []\n\n    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}\n    predecessor_position_um: dict[int, np.ndarray] = {}\n    selected_edges: list[dict[str, object]] = []\n\n    def assign_pass(\n        source_ids: list[int],\n        target_ids: list[int],\n        gate_um: float,\n    ) -> list[tuple[int, int, float, float, float]]:\n        if not source_ids or not target_ids:\n            return []\n        big = gate_um * 1000.0 + 1.0\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\n        raw_dist = np.full_like(cost, np.inf)\n        motion_dist = np.full_like(cost, np.inf)\n        prob_matrix = np.zeros_like(cost)\n        for i, source_id in enumerate(source_ids):\n            source_pos = position_um[source_id]\n            prev_pos = predecessor_position_um.get(source_id)\n            if prev_pos is None:\n                predicted = source_pos\n            else:\n                stats["wave_velocity_consumed"] = float(MOTION_RELINK_VELOCITY_WEIGHT)\n                stats["wave_velocity_calls"] = stats.get("wave_velocity_calls", 0) + 1\n                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n            for j, target_id in enumerate(target_ids):\n                target_pos = position_um[target_id]\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > gate_um:\n                    continue\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                raw_dist[i, j] = raw\n                motion_dist[i, j] = motion\n                prob_matrix[i, j] = prob\n                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob\n        row_ind, col_ind = linear_sum_assignment(cost)\n        matches: list[tuple[int, int, float, float, float]] = []\n        for r, c in zip(row_ind, col_ind):\n            if cost[r, c] >= big:\n                continue\n            matches.append((\n                source_ids[int(r)],\n                target_ids[int(c)],\n                float(raw_dist[r, c]),\n                float(motion_dist[r, c]),\n                float(prob_matrix[r, c]),\n            ))\n        return matches\n\n    times = sorted(ids_by_t)\n    for t in times:\n        source_ids = ids_by_t.get(t, [])\n        target_ids = ids_by_t.get(t + 1, [])\n        if not source_ids or not target_ids:\n            continue\n        unmatched_sources = set(source_ids)\n        unmatched_targets = set(target_ids)\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\n        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\n            for source_id, target_id, raw, motion, prob in matches:\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    continue\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\n                if pass_name == "tight":\n                    stats["motion_relink_tight_edges"] += 1\n                else:\n                    stats["motion_relink_relaxed_edges"] += 1\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\n            selected_edges.append({\n                "source_id": source_id,\n                "target_id": target_id,\n                "edge_prob": prob,\n                "distance_um": raw,\n                "motion_distance_um": motion,\n                "_wave_model_probability": scored_probability(learned_edge_probs[(source_id, target_id)]) if (source_id, target_id) in learned_edge_probs else None,\n                "motion_relinked": 1,\n                "motion_pass": pass_name,\n            })\n            predecessor_position_um[target_id] = position_um[source_id]\n        stats["motion_relink_frames"] += 1\n\n    stats["motion_relink_edges"] = len(selected_edges)\n    return selected_edges\n\ndef close_single_frame_gaps(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    incident = outgoing | incoming\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    isolated_by_t: dict[int, list[int]] = {}\n    all_ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        all_ids_by_t.setdefault(t, []).append(node_id)\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incident:\n            isolated_by_t.setdefault(t, []).append(node_id)\n\n    max_synthetic = min(\n        GAP_CLOSE_MAX_ADDED_ABS,\n        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,\n    )\n    next_id = _next_node_id(nodes_by_id)\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n    used_starts: set[int] = set()\n    used_isolated: set[int] = set()\n    synthetic_added = 0\n    new_edges: list[dict[str, object]] = []\n\n    density_cache: dict[int, dict[int, float]] = {}\n\n    def frame_local_spacing(t: int) -> dict[int, float]:\n        cached = density_cache.get(t)\n        if cached is not None:\n            return cached\n\n        frame_ids = all_ids_by_t.get(t, [])\n        if len(frame_ids) <= 1:\n            result = {\n                node_id: GAP_DENSITY_REFERENCE_UM\n                for node_id in frame_ids\n            }\n            density_cache[t] = result\n            return result\n\n        positions = np.stack(\n            [_position_um(nodes_by_id[node_id]) for node_id in frame_ids]\n        )\n        tree = cKDTree(positions)\n        query_k = min(\n            len(frame_ids),\n            max(2, GAP_DENSITY_NEIGHBORS + 1),\n        )\n        distances, _ = tree.query(positions, k=query_k)\n        if distances.ndim == 1:\n            distances = distances[:, None]\n\n        result: dict[int, float] = {}\n        for idx, node_id in enumerate(frame_ids):\n            neighbour_distances = distances[idx, 1:]\n            neighbour_distances = neighbour_distances[\n                np.isfinite(neighbour_distances)\n            ]\n            spacing = (\n                float(np.median(neighbour_distances))\n                if neighbour_distances.size\n                else GAP_DENSITY_REFERENCE_UM\n            )\n            result[node_id] = spacing\n\n        density_cache[t] = result\n        stats["gap_density_nodes_scored"] += len(result)\n        return result\n\n    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)\n    stats["gap_close_effective_max_gap"] = effective_gap_max\n    for gap in range(1, effective_gap_max + 1):\n        for t, end_ids in sorted(ends_by_t.items()):\n            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]\n            if not end_ids or not start_ids:\n                continue\n\n            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]\n            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]\n            threshold_um = GAP_CLOSE_UM * (gap + 1)\n            d = np.zeros(\n                (len(end_ids), len(start_ids)),\n                dtype=np.float64,\n            )\n            adaptive_threshold = np.full_like(d, threshold_um)\n\n            source_spacing = frame_local_spacing(t)\n            target_spacing = frame_local_spacing(t + gap + 1)\n\n            for i, ep in enumerate(end_points):\n                for j, sp in enumerate(start_points):\n                    d[i, j] = point_distance_um(ep, sp)\n\n                    if GAP_DENSITY_ADAPTIVE:\n                        local_spacing = 0.5 * (\n                            source_spacing.get(\n                                end_ids[i],\n                                GAP_DENSITY_REFERENCE_UM,\n                            )\n                            + target_spacing.get(\n                                start_ids[j],\n                                GAP_DENSITY_REFERENCE_UM,\n                            )\n                        )\n                        step_delta = float(\n                            np.clip(\n                                GAP_DENSITY_GAIN\n                                * (\n                                    local_spacing\n                                    - GAP_DENSITY_REFERENCE_UM\n                                ),\n                                -GAP_DENSITY_MAX_STEP_DELTA_UM,\n                                GAP_DENSITY_MAX_STEP_DELTA_UM,\n                            )\n                        )\n                        adaptive_threshold[i, j] = (\n                            threshold_um + step_delta * (gap + 1)\n                        )\n                        stats[\n                            "gap_density_step_delta_milli_sum"\n                        ] += int(round(1000.0 * step_delta))\n\n            base_allowed = d <= threshold_um\n            adaptive_allowed = d <= adaptive_threshold\n\n            stats["gap_density_candidates_expanded"] += int(\n                (adaptive_allowed & ~base_allowed).sum()\n            )\n            stats["gap_density_candidates_restricted"] += int(\n                (base_allowed & ~adaptive_allowed).sum()\n            )\n            stats["gap_candidates"] += int(adaptive_allowed.sum())\n\n            if not np.isfinite(d).any():\n                continue\n\n            max_threshold = float(np.max(adaptive_threshold))\n            big = max_threshold * 1000.0 + 1.0\n            cost = np.where(adaptive_allowed, d, big)\n            row_ind, col_ind = linear_sum_assignment(cost)\n\n            for r, c in zip(row_ind, col_ind):\n                if not adaptive_allowed[r, c]:\n                    continue\n                if not base_allowed[r, c]:\n                    stats[\n                        "gap_density_selected_outside_base"\n                    ] += 1\n                source_id = end_ids[int(r)]\n                target_id = start_ids[int(c)]\n                if source_id in outgoing or target_id in used_starts:\n                    continue\n\n                source = nodes_by_id[source_id]\n                target = nodes_by_id[target_id]\n                mid_t = int(source["t"]) + gap\n                mid_point = (\n                    (float(source["z"]) + float(target["z"])) / 2.0,\n                    (float(source["y"]) + float(target["y"])) / 2.0,\n                    (float(source["x"]) + float(target["x"])) / 2.0,\n                )\n\n                middle_id: int | None = None\n                middle_reused = False\n                if GAP_CLOSE_REUSE_EXISTING:\n                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]\n                    if candidates:\n                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]\n                        best_idx = int(np.argmin(distances))\n                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:\n                            middle_id = candidates[best_idx]\n                            middle_reused = True\n\n                if middle_id is None:\n                    if synthetic_added >= max_synthetic:\n                        stats["gap_skipped_node_cap"] += 1\n                        continue\n                    middle_id = next_id\n                    next_id += 1\n                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)\n                    nodes_by_id[middle_id] = {\n                        "node_id": middle_id,\n                        "t": mid_t,\n                        "z": refined_point[0],\n                        "y": refined_point[1],\n                        "x": refined_point[2],\n                        "gap_synthetic": 1,\n                    }\n                    synthetic_added += 1\n                    stats["gap_inserted_synthetic"] += 1\n\n                middle = nodes_by_id[middle_id]\n                gap_span_um = float(d[r, c])\n                marginal_gap = gap_span_um >= DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM\n                synthetic_middle = int(middle.get("gap_synthetic", 0)) == 1\n                requires_center_confirmation = (\n                    DEEPCENTER_GAP_VETO and marginal_gap and synthetic_middle\n                )\n                if DEEPCENTER_GAP_VETO and not marginal_gap:\n                    stats["deepcenter_gap_bypassed_strong_motion"] += 1\n                elif DEEPCENTER_GAP_VETO and not synthetic_middle:\n                    stats["deepcenter_gap_bypassed_observed_node"] += 1\n                if requires_center_confirmation and not deepcenter_accept_repair_point(\n                    dataset,\n                    mid_t,\n                    node_point(middle),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "gap",\n                    DEEPCENTER_GAP_THRESHOLD,\n                ):\n                    if int(middle.get("gap_synthetic", 0)) == 1:\n                        nodes_by_id.pop(middle_id, None)\n                        synthetic_added = max(0, synthetic_added - 1)\n                        stats["gap_inserted_synthetic"] = max(0, stats["gap_inserted_synthetic"] - 1)\n                    continue\n                if middle_reused:\n                    used_isolated.add(middle_id)\n                    stats["gap_reused_existing"] += 1\n\n                e1 = {\n                    "source_id": source_id,\n                    "target_id": middle_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(source, middle),\n                    "gap_closed": 1,\n                }\n                e2 = {\n                    "source_id": middle_id,\n                    "target_id": target_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(middle, target),\n                    "gap_closed": 1,\n                }\n                new_edges.extend([e1, e2])\n                outgoing.add(source_id)\n                incoming.add(middle_id)\n                outgoing.add(middle_id)\n                incoming.add(target_id)\n                used_starts.add(target_id)\n                stats["gap_pairs_selected"] += 1\n                stats["gap_added_edges"] += 2\n\n    if new_edges:\n        edges = [*edges, *new_edges]\n    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]\n    return nodes_by_id, edges\n\n\ndef _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_source: dict[int, list[int]] = {}\n    for edge in edges:\n        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))\n    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}\n\n\ndef _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_target: dict[int, list[int]] = {}\n    for edge in edges:\n        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))\n    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}\n\n\ndef recover_strict_gap2(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    predecessor = _single_predecessor_map(edges)\n    successor = _single_successor_map(edges)\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n\n    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))\n    proposals: list[tuple[float, int, int, int, float]] = []\n\n    def pos_um(node_id: int) -> np.ndarray:\n        node = nodes_by_id[node_id]\n        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)\n\n    for t, end_ids in sorted(ends_by_t.items()):\n        start_ids = starts_by_t.get(t + 3, [])\n        if not end_ids or not start_ids:\n            continue\n        for end_id in end_ids:\n            end_pos = pos_um(end_id)\n            for start_id in start_ids:\n                start_pos = pos_um(start_id)\n                dist = float(np.linalg.norm(start_pos - end_pos))\n                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:\n                    continue\n                step = (start_pos - end_pos) / 3.0\n                context_penalty = 0.0\n                if GAP2_REQUIRE_CONTEXT:\n                    ok_context = False\n                    prev_id = predecessor.get(end_id)\n                    if prev_id is not None:\n                        prev_step = end_pos - pos_um(prev_id)\n                        prev_norm = float(np.linalg.norm(prev_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if prev_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    next_id = successor.get(start_id)\n                    if next_id is not None:\n                        next_step = pos_um(next_id) - start_pos\n                        next_norm = float(np.linalg.norm(next_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if next_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    if not ok_context:\n                        continue\n                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))\n\n    proposals.sort(key=lambda item: item[0])\n    stats["gap2_candidates"] = len(proposals)\n    if not proposals:\n        return nodes_by_id, edges\n\n    selected: list[tuple[float, int, int, int, float]] = []\n    used_ends: set[int] = set()\n    used_starts: set[int] = set()\n    per_frame_count: dict[int, int] = {}\n    for proposal in proposals:\n        if len(selected) >= cap:\n            stats["gap2_skipped_cap"] += 1\n            break\n        _, end_id, start_id, t, _ = proposal\n        if end_id in used_ends or start_id in used_starts:\n            continue\n        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))\n        if per_frame_count.get(t, 0) >= frame_cap:\n            continue\n        selected.append(proposal)\n        used_ends.add(end_id)\n        used_starts.add(start_id)\n        per_frame_count[t] = per_frame_count.get(t, 0) + 1\n\n    if not selected:\n        return nodes_by_id, edges\n\n    next_node_id = _next_node_id(nodes_by_id)\n    frame_cache: dict[int, np.ndarray] = {}\n    new_edges: list[dict[str, object]] = []\n    for _, end_id, start_id, t, _ in selected:\n        source = nodes_by_id[end_id]\n        target = nodes_by_id[start_id]\n        previous_id = end_id\n        inserted_ids: list[int] = []\n        for k in (1, 2):\n            frac = k / 3.0\n            mid_t = int(source["t"]) + k\n            midpoint = (\n                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,\n                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,\n                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,\n            )\n            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)\n            node_id = next_node_id\n            next_node_id += 1\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": mid_t,\n                "z": refined_point[0],\n                "y": refined_point[1],\n                "x": refined_point[2],\n            }\n            inserted_ids.append(node_id)\n            current = nodes_by_id[node_id]\n            new_edges.append({\n                "source_id": previous_id,\n                "target_id": node_id,\n                "edge_prob": None,\n                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),\n                "gap2_recovered": 1,\n            })\n            previous_id = node_id\n        new_edges.append({\n            "source_id": previous_id,\n            "target_id": start_id,\n            "edge_prob": None,\n            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),\n            "gap2_recovered": 1,\n        })\n        stats["gap2_pairs_selected"] += 1\n        stats["gap2_added_nodes"] += len(inserted_ids)\n        stats["gap2_added_edges"] += 3\n\n    return nodes_by_id, [*edges, *new_edges]\n\n\ndef add_safe_divisions_postlink(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\n        return edges\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n \n    out_by_source: dict[int, list[dict[str, object]]] = {}\n    incoming: set[int] = set()\n    for edge in edges:\n        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)\n        incoming.add(int(edge["target_id"]))\n \n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n \n    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}\n    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))\n    added: list[dict[str, object]] = []\n    used_targets: set[int] = set()\n    used_sources: set[int] = set()  \n \n    for t in sorted(ids_by_t):\n        child_frame_ids = ids_by_t.get(t + 1, [])\n        if not child_frame_ids:\n            continue\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\n        if not source_ids or not candidate_ids:\n            continue\n \n        \n        \n        \n        \n        \n        candidate_tree = None\n        if SAFE_DIV_REQUIRE_MUTUAL_NN:\n            candidate_positions = np.stack([_position_um(nodes_by_id[cid]) for cid in candidate_ids])\n            candidate_tree = cKDTree(candidate_positions)\n \n        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))\n        proposals: list[tuple[float, int, int, float, float]] = []\n        for source_id in source_ids:\n            source = nodes_by_id[source_id]\n            existing_child_edge = out_by_source[source_id][0]\n            existing_child_id = int(existing_child_edge["target_id"])\n            existing_child = nodes_by_id.get(existing_child_id)\n            if existing_child is None or int(existing_child["t"]) != t + 1:\n                continue\n            child_dist = edge_distance_um(source, existing_child)\n            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:\n                continue\n \n            \n            \n            \n            \n            mutual_nn_id = None\n            if candidate_tree is not None:\n                _, nn_idx = candidate_tree.query(_position_um(existing_child))\n                mutual_nn_id = candidate_ids[int(nn_idx)]\n \n            for candidate_id in candidate_ids:\n                if (source_id, candidate_id) in existing_edges:\n                    continue\n                candidate = nodes_by_id[candidate_id]\n                parent_dist = edge_distance_um(source, candidate)\n                if parent_dist > SAFE_DIV_MAX_UM:\n                    continue\n                sister_dist = edge_distance_um(existing_child, candidate)\n                if sister_dist > SAFE_DIV_SISTER_MAX_UM:\n                    continue\n \n                \n                if SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_nn_id:\n                    stats["safe_division_mutual_nn_rejected"] += 1\n                    continue\n \n                \n                \n                \n                \n                if SAFE_DIV_REQUIRE_DIVERGENCE:\n                    c1_succ = out_by_source.get(existing_child_id, [])\n                    q_succ = out_by_source.get(candidate_id, [])\n                    if len(c1_succ) != 1 or len(q_succ) != 1:\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n                    c1_grandchild = nodes_by_id.get(int(c1_succ[0]["target_id"]))\n                    q_grandchild = nodes_by_id.get(int(q_succ[0]["target_id"]))\n                    if (\n                        c1_grandchild is None or q_grandchild is None\n                        or int(c1_grandchild["t"]) != t + 2\n                        or int(q_grandchild["t"]) != t + 2\n                    ):\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n                    grandchild_dist = edge_distance_um(c1_grandchild, q_grandchild)\n                    if grandchild_dist - sister_dist < SAFE_DIV_DIVERGE_UM:\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n \n                stats["safe_division_geometric_candidates"] += 1\n                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\n                    dataset,\n                    int(candidate["t"]),\n                    node_point(candidate),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "safe_div",\n                    DEEPCENTER_SAFE_DIV_THRESHOLD,\n                ):\n                    continue\n                \n                \n                \n                \n                \n                \n                if SAFE_DIV_SISTER_SYMMETRY_TAU > 0.0:\n                    _sym_denom = max((child_dist + parent_dist) / 2.0, 1e-6)\n                    if abs(child_dist - parent_dist) / _sym_denom > SAFE_DIV_SISTER_SYMMETRY_TAU:\n                        stats["safe_division_symmetry_rejected"] += 1\n                        continue\n                score = parent_dist + 0.15 * sister_dist\n                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\n \n        stats["safe_division_candidates"] += len(proposals)\n        if not proposals:\n            continue\n        proposals.sort(key=lambda item: item[0])\n        added_this_frame = 0\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\n            if len(added) >= global_cap:\n                stats["safe_division_skipped_cap"] += 1\n                break\n            if added_this_frame >= frame_cap:\n                break\n            if candidate_id in used_targets or candidate_id in incoming:\n                continue\n            if source_id in used_sources:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            added.append({\n                "source_id": source_id,\n                "target_id": candidate_id,\n                "edge_prob": None,\n                "distance_um": parent_dist,\n                "safe_division": 1,\n            })\n            used_targets.add(candidate_id)\n            used_sources.add(source_id)\n            added_this_frame += 1\n \n    if added:\n        stats["safe_divisions_added"] = len(added)\n        return [*edges, *added]\n    return edges\n_sprint_original_safe_div=add_safe_divisions_postlink\ndef add_safe_divisions_postlink(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n    frame_cache: dict[int, np.ndarray] | None = None,\n    deepcenter_cache: dict[tuple[str, int], np.ndarray] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\n        return edges\n    frame_cache = frame_cache if frame_cache is not None else {}\n    deepcenter_cache = deepcenter_cache if deepcenter_cache is not None else {}\n \n    out_by_source: dict[int, list[dict[str, object]]] = {}\n    incoming: set[int] = set()\n    for edge in edges:\n        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)\n        incoming.add(int(edge["target_id"]))\n \n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n \n    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}\n    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))\n    added: list[dict[str, object]] = []\n    used_targets: set[int] = set()\n    used_sources: set[int] = set()  \n \n    for t in sorted(ids_by_t):\n        child_frame_ids = ids_by_t.get(t + 1, [])\n        if not child_frame_ids:\n            continue\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\n        if not source_ids or not candidate_ids:\n            continue\n \n        \n        \n        \n        \n        \n        candidate_tree = None\n        if SAFE_DIV_REQUIRE_MUTUAL_NN:\n            candidate_positions = np.stack([_position_um(nodes_by_id[cid]) for cid in candidate_ids])\n            candidate_tree = cKDTree(candidate_positions)\n \n        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))\n        proposals: list[tuple[float, int, int, float, float]] = []\n        for source_id in source_ids:\n            source = nodes_by_id[source_id]\n            existing_child_edge = out_by_source[source_id][0]\n            existing_child_id = int(existing_child_edge["target_id"])\n            existing_child = nodes_by_id.get(existing_child_id)\n            if existing_child is None or int(existing_child["t"]) != t + 1:\n                continue\n            child_dist = edge_distance_um(source, existing_child)\n            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:\n                continue\n \n            \n            \n            \n            \n            mutual_nn_id = None\n            if candidate_tree is not None:\n                _, nn_idx = candidate_tree.query(_position_um(existing_child))\n                mutual_nn_id = candidate_ids[int(nn_idx)]\n \n            for candidate_id in candidate_ids:\n                if (source_id, candidate_id) in existing_edges:\n                    continue\n                candidate = nodes_by_id[candidate_id]\n                parent_dist = edge_distance_um(source, candidate)\n                if parent_dist > SAFE_DIV_MAX_UM:\n                    continue\n                sister_dist = edge_distance_um(existing_child, candidate)\n                if sister_dist > SAFE_DIV_SISTER_MAX_UM:\n                    continue\n \n                \n                if SAFE_DIV_REQUIRE_MUTUAL_NN and candidate_id != mutual_nn_id:\n                    stats["safe_division_mutual_nn_rejected"] += 1\n                    continue\n \n                \n                \n                \n                \n                if SAFE_DIV_REQUIRE_DIVERGENCE:\n                    c1_succ = out_by_source.get(existing_child_id, [])\n                    q_succ = out_by_source.get(candidate_id, [])\n                    if len(c1_succ) != 1 or len(q_succ) != 1:\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n                    c1_grandchild = nodes_by_id.get(int(c1_succ[0]["target_id"]))\n                    q_grandchild = nodes_by_id.get(int(q_succ[0]["target_id"]))\n                    if (\n                        c1_grandchild is None or q_grandchild is None\n                        or int(c1_grandchild["t"]) != t + 2\n                        or int(q_grandchild["t"]) != t + 2\n                    ):\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n                    grandchild_dist = edge_distance_um(c1_grandchild, q_grandchild)\n                    if grandchild_dist - sister_dist < SAFE_DIV_DIVERGE_UM:\n                        stats["safe_division_divergence_rejected"] += 1\n                        continue\n \n                stats["safe_division_geometric_candidates"] += 1\n                if DEEPCENTER_SAFE_DIV_VETO and not deepcenter_accept_repair_point(\n                    dataset,\n                    int(candidate["t"]),\n                    node_point(candidate),\n                    deepcenter_bundle,\n                    frame_cache,\n                    deepcenter_cache,\n                    stats,\n                    "safe_div",\n                    DEEPCENTER_SAFE_DIV_THRESHOLD,\n                ):\n                    continue\n                \n                \n                \n                \n                \n                \n                if SAFE_DIV_SISTER_SYMMETRY_TAU > 0.0:\n                    _sym_denom = max((child_dist + parent_dist) / 2.0, 1e-6)\n                    if abs(child_dist - parent_dist) / _sym_denom > SAFE_DIV_SISTER_SYMMETRY_TAU:\n                        stats["safe_division_symmetry_rejected"] += 1\n                        continue\n                score = parent_dist + 0.15 * sister_dist\n                proposal = (score, source_id, candidate_id, parent_dist, sister_dist)\n                if SPRINT_POLICY.admit((nodes_by_id, out_by_source), proposal, dataset, t, existing_child_id):\n                    proposals.append(proposal)\n \n        stats["safe_division_candidates"] += len(proposals)\n        if not proposals:\n            continue\n        SPRINT_POLICY.order(proposals, dataset, t)\n        added_this_frame = 0\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\n            if len(added) >= global_cap:\n                stats["safe_division_skipped_cap"] += 1\n                break\n            if added_this_frame >= frame_cap:\n                break\n            if candidate_id in used_targets or candidate_id in incoming:\n                continue\n            if source_id in used_sources:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            added.append({\n                "source_id": source_id,\n                "target_id": candidate_id,\n                "edge_prob": None,\n                "distance_um": parent_dist,\n                "safe_division": 1,\n            })\n            SPRINT_POLICY.selected(dataset, t, source_id, candidate_id)\n            used_targets.add(candidate_id)\n            used_sources.add(source_id)\n            added_this_frame += 1\n \n    if added:\n        stats["safe_divisions_added"] = len(added)\n        return [*edges, *added]\n    return edges\n_sprint_patched_safe_div=add_safe_divisions_postlink\n\n\n\ndef filter_short_track_components(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:\n        return nodes_by_id, edges\n\n    parent = {node_id: node_id for node_id in nodes_by_id}\n\n    def find(node_id: int) -> int:\n        while parent[node_id] != node_id:\n            parent[node_id] = parent[parent[node_id]]\n            node_id = parent[node_id]\n        return node_id\n\n    def union(a: int, b: int) -> None:\n        if a not in parent or b not in parent:\n            return\n        ra = find(a)\n        rb = find(b)\n        if ra != rb:\n            parent[ra] = rb\n\n    out_count: dict[int, int] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        union(source_id, target_id)\n        out_count[source_id] = out_count.get(source_id, 0) + 1\n\n    components: dict[int, list[int]] = {}\n    for node_id in nodes_by_id:\n        components.setdefault(find(node_id), []).append(node_id)\n\n    component_edges: dict[int, list[dict[str, object]]] = {root: [] for root in components}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        if source_id in parent and target_id in parent:\n            component_edges.setdefault(find(source_id), []).append(edge)\n\n    keep: set[int] = set()\n    for root, members in components.items():\n        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)\n        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):\n            keep.update(members)\n\n    if not keep:\n        stats["short_track_filter_skipped_all"] += 1\n        return nodes_by_id, edges\n\n    removed_before_rescue = len(nodes_by_id) - len(keep)\n    _trio_rescue_row = {"removed_before_rescue": removed_before_rescue, "removed_frac": removed_before_rescue / max(len(nodes_by_id), 1), "threshold": SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC, "triggered": False, "budget": 0, "rescued_nodes": 0}\n    _trio_rescue_rows.append(_trio_rescue_row)\n    if removed_before_rescue <= 0:\n        return nodes_by_id, edges\n\n    if ADAPTIVE_SHORT_TRACK_RESCUE:\n        removed_frac = removed_before_rescue / max(len(nodes_by_id), 1)\n        if removed_frac >= SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC:\n            budget = min(\n                SHORT_TRACK_RESCUE_MAX_NODES_ABS,\n                max(0, int(round(len(nodes_by_id) * SHORT_TRACK_RESCUE_MAX_NODES_FRAC))),\n            )\n            _trio_rescue_row.update(triggered=True, budget=budget)\n            stats["short_track_rescue_triggered"] = 1\n            stats["short_track_rescue_budget"] = budget\n            proposals: list[tuple[float, int, float, int, list[int]]] = []\n            for root, members in components.items():\n                if set(members) & keep:\n                    continue\n                if len(members) < SHORT_TRACK_RESCUE_MIN_LEN or len(members) >= OUTPUT_MIN_TRACK_LEN:\n                    continue\n                c_edges = component_edges.get(root, [])\n                if not c_edges:\n                    continue\n                probs: list[float] = []\n                dists: list[float] = []\n                for edge in c_edges:\n                    try:\n                        prob = float(edge.get("edge_prob", 0.0))\n                    except (TypeError, ValueError):\n                        prob = 0.0\n                    if np.isfinite(prob):\n                        probs.append(prob)\n                    try:\n                        dist = float(edge.get("distance_um", np.nan))\n                    except (TypeError, ValueError):\n                        dist = np.nan\n                    if np.isfinite(dist):\n                        dists.append(dist)\n                mean_prob = float(np.mean(probs)) if probs else 0.0\n                mean_dist = float(np.mean(dists)) if dists else float("inf")\n                if mean_prob < SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB:\n                    continue\n                if mean_dist > SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM:\n                    continue\n                score = mean_prob - 0.02 * mean_dist + 0.004 * len(members)\n                proposals.append((score, len(members), mean_prob, root, members))\n            proposals.sort(reverse=True)\n            rescued_nodes = 0\n            rescued_components = 0\n            for _, size, _, _, members in proposals:\n                if budget <= 0 or rescued_nodes + size > budget:\n                    continue\n                keep.update(members)\n                rescued_nodes += size\n                rescued_components += 1\n            stats["short_track_rescue_components"] = rescued_components\n            _trio_rescue_row["rescued_nodes"] = rescued_nodes\n            stats["short_track_rescue_nodes"] = rescued_nodes\n\n    removed_nodes = len(nodes_by_id) - len(keep)\n    if removed_nodes <= 0:\n        return nodes_by_id, edges\n\n    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}\n    kept_edges = [\n        edge for edge in edges\n        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes\n    ]\n    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))\n    stats["short_track_nodes_removed"] = removed_nodes\n    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)\n    return kept_nodes, kept_edges\n\n\ndef linefit_smooth_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> dict[int, dict[str, object]]:\n    """Smooth linear track interiors without changing graph topology."""\n    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:\n        return nodes_by_id\n\n    predecessor: dict[int, list[int]] = {}\n    successor: dict[int, list[int]] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        source = nodes_by_id.get(source_id)\n        target = nodes_by_id.get(target_id)\n        if source is None or target is None:\n            continue\n        if int(target["t"]) != int(source["t"]) + 1:\n            continue\n        successor.setdefault(source_id, []).append(target_id)\n        predecessor.setdefault(target_id, []).append(source_id)\n\n    original_pos = {\n        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)\n        for node_id, node in nodes_by_id.items()\n    }\n    updated_pos: dict[int, np.ndarray] = {}\n    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))\n\n    for node_id in sorted(nodes_by_id):\n        neighbourhood: list[tuple[int, int]] = [(0, node_id)]\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            prev_ids = predecessor.get(current, [])\n            if len(prev_ids) != 1:\n                break\n            current = prev_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((-step, current))\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            next_ids = successor.get(current, [])\n            if len(next_ids) != 1:\n                break\n            current = next_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((step, current))\n\n        if len(neighbourhood) < 3:\n            stats["linefit_skipped_nodes"] += 1\n            continue\n\n        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)\n        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])\n        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)\n        if not np.isfinite(fitted).all():\n            stats["linefit_skipped_nodes"] += 1\n            continue\n        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted\n\n    for node_id, pos in updated_pos.items():\n        nodes_by_id[node_id]["z"] = float(pos[0])\n        nodes_by_id[node_id]["y"] = float(pos[1])\n        nodes_by_id[node_id]["x"] = float(pos[2])\n\n    stats["linefit_smoothed_nodes"] = len(updated_pos)\n    return nodes_by_id\n\n\ndef filter_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    raw_edges: list[dict[str, object]],\n    dataset: str | None = None,\n    deepcenter_bundle: dict[str, object] | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\n    stats = {\n        "raw_edges": len(raw_edges),\n        "dropped_nonconsecutive_edges": 0,\n        "dropped_long_edges": 0,\n        "dropped_multi_parent_edges": 0,\n        "dropped_multi_child_edges": 0,\n        "dropped_division_edges": 0,\n        "gap_candidates": 0,\n        "gap_pairs_selected": 0,\n        "gap_reused_existing": 0,\n        "gap_inserted_synthetic": 0,\n        "gap_added_nodes": 0,\n        "gap_added_edges": 0,\n        "gap_skipped_node_cap": 0,\n        "gap_density_nodes_scored": 0,\n        "gap_density_candidates_expanded": 0,\n        "gap_density_candidates_restricted": 0,\n        "gap_density_selected_outside_base": 0,\n        "gap_density_step_delta_milli_sum": 0,\n        "gap_refined_synthetic": 0,\n        "gap_refine_failed": 0,\n        "gap_refine_rejected_shift": 0,\n        "pruned_isolated_nodes": 0,\n        "motion_relink_edges": 0,\n        "motion_relink_tight_edges": 0,\n        "motion_relink_relaxed_edges": 0,\n        "motion_relink_frames": 0,\n        "motion_relink_replaced_raw_edges": 0,\n        "motion_relink_fallback_raw": 0,\n        "motion_relink_skipped_large_frame": 0,\n        "gap2_candidates": 0,\n        "gap2_pairs_selected": 0,\n        "gap2_added_nodes": 0,\n        "gap2_added_edges": 0,\n        "gap2_skipped_cap": 0,\n        "safe_division_candidates": 0,\n        "safe_division_geometric_candidates": 0,  \n        "safe_divisions_added": 0,\n        "safe_division_skipped_cap": 0,\n        "safe_division_mutual_nn_rejected": 0,\n        "safe_division_divergence_rejected": 0,\n        "safe_division_symmetry_rejected": 0,  \n        "deepcenter_gap_checked": 0,\n        "deepcenter_gap_bypassed_strong_motion": 0,\n        "deepcenter_gap_bypassed_observed_node": 0,\n        "deepcenter_gap_accepted": 0,\n        "deepcenter_gap_rejected": 0,\n        "deepcenter_gap_missing": 0,\n        "deepcenter_safe_div_checked": 0,\n        "deepcenter_safe_div_accepted": 0,\n        "deepcenter_safe_div_rejected": 0,\n        "deepcenter_safe_div_missing": 0,\n        "short_track_components_removed": 0,\n        "short_track_nodes_removed": 0,\n        "short_track_edges_removed": 0,\n        "short_track_filter_skipped_all": 0,\n        "short_track_rescue_triggered": 0,\n        "short_track_rescue_components": 0,\n        "short_track_rescue_nodes": 0,\n        "short_track_rescue_budget": 0,\n        "linefit_smoothed_nodes": 0,\n        "linefit_skipped_nodes": 0,\n    }\n\n    edges: list[dict[str, object]] = []\n    for edge in raw_edges:\n        edge["_wave_model_probability"] = scored_probability(edge.get("edge_prob"))\n        source = nodes_by_id.get(int(edge["source_id"]))\n        target = nodes_by_id.get(int(edge["target_id"]))\n        if source is None or target is None:\n            continue\n        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:\n            stats["dropped_nonconsecutive_edges"] += 1\n            continue\n        distance_um = edge_distance_um(source, target)\n        edge["distance_um"] = distance_um\n        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:\n            stats["dropped_long_edges"] += 1\n            continue\n        edges.append(edge)\n\n    if OUTPUT_MOTION_RELINK:\n        learned_edge_probs: dict[tuple[int, int], float] = {}\n        for edge in edges:\n            prob = edge.get("edge_prob")\n            if prob is None:\n                continue\n            try:\n                prob = float(prob)\n            except (TypeError, ValueError):\n                continue\n            if np.isfinite(prob):\n                key = (int(edge["source_id"]), int(edge["target_id"]))\n                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)\n        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)\n        if motion_edges:\n            stats["motion_relink_replaced_raw_edges"] = len(edges)\n            edges = motion_edges\n        else:\n            stats["motion_relink_fallback_raw"] = 1\n\n    if OUTPUT_SINGLE_PARENT_REPAIR and edges:\n        best_by_target: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            target_id = int(edge["target_id"])\n            prev = best_by_target.get(target_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_target[target_id] = edge\n        kept_ids = {id(edge) for edge in best_by_target.values()}\n        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    if OUTPUT_SINGLE_CHILD_REPAIR and edges:\n        best_by_source: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            prev = best_by_source.get(source_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_source[source_id] = edge\n        kept_ids = {id(edge) for edge in best_by_source.values()}\n        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    print(f"  [{dataset}] after edge-filter+motion-relink: {len(nodes_by_id)} nodes, {len(edges)} edges")\n    repair_frame_cache: dict[int, np.ndarray] = {}\n    deepcenter_heatmap_cache: dict[tuple[str, int], np.ndarray] = {}\n    nodes_by_id, edges = close_single_frame_gaps(\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)\n    print(f"  [{dataset}] after gap-closing (single-frame + gap2): {len(nodes_by_id)} nodes, {len(edges)} edges")\n    edges = add_safe_divisions_postlink(\n        nodes_by_id,\n        edges,\n        stats,\n        dataset=dataset,\n        deepcenter_bundle=deepcenter_bundle,\n        frame_cache=repair_frame_cache,\n        deepcenter_cache=deepcenter_heatmap_cache,\n    )\n\n    _geo_cands = stats[\'safe_division_geometric_candidates\']\n    _post_veto_cands = stats[\'safe_division_candidates\']\n    _rejected_by_dc = _geo_cands - _post_veto_cands\n    print(\n        f"  [{dataset}] after safe-division repair: {len(nodes_by_id)} nodes, {len(edges)} edges"\n        f" (geometric_candidates={_geo_cands}, deepcenter_rejected={_rejected_by_dc},"\n        f" post_veto_candidates={_post_veto_cands}, added={stats[\'safe_divisions_added\']},"\n        f" cap_skipped={stats[\'safe_division_skipped_cap\']},"\n        f" mutual_nn_rejected={stats[\'safe_division_mutual_nn_rejected\']},"\n        f" divergence_rejected={stats[\'safe_division_divergence_rejected\']})"\n    )\n    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:\n        by_source: dict[int, list[dict[str, object]]] = {}\n        for edge in edges:\n            by_source.setdefault(int(edge["source_id"]), []).append(edge)\n\n        filtered: list[dict[str, object]] = []\n        for source_id, source_edges in by_source.items():\n            if len(source_edges) <= 1:\n                filtered.extend(source_edges)\n                continue\n\n            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)\n            source = nodes_by_id[source_id]\n            top1 = ranked[0]\n            top2 = ranked[1]\n            d1 = float(top1["distance_um"])\n            d2 = float(top2["distance_um"])\n            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])\n            valid_division = (\n                max(d1, d2) <= DIV_PARENT_MAX_UM\n                and sister <= DIV_SISTER_MAX_UM\n                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1\n                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1\n            )\n            if valid_division:\n                filtered.extend([top1, top2])\n                stats["dropped_division_edges"] += max(0, len(ranked) - 2)\n            elif DIV_DROP_TO_SINGLE_IF_BAD:\n                filtered.append(top1)\n                stats["dropped_division_edges"] += len(ranked) - 1\n            else:\n                filtered.extend(ranked)\n        edges = filtered\n\n    if OUTPUT_PRUNE_ISOLATED:\n        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}\n        if incident:\n            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}\n            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)\n            nodes_by_id = kept_nodes\n            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]\n\n    print(f"  [{dataset}] after division-geometry-filter+prune-isolated: {len(nodes_by_id)} nodes, {len(edges)} edges")\n    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)\n    print(f"  [{dataset}] after short-track filtering: {len(nodes_by_id)} nodes, {len(edges)} edges"\n          f" (components_removed={stats[\'short_track_components_removed\']})")\n    nodes_by_id, edges, leaf_stats = prune_leaves(nodes_by_id, edges, wave_last_frame(dataset) if WAVE_LEAF_THRESHOLD is not None else None, WAVE_LEAF_THRESHOLD)\n    stats["wave_leaf"] = leaf_stats\n    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)\n    print(f"  [{dataset}] FINAL: {len(nodes_by_id)} nodes, {len(edges)} edges")\n\n    return nodes_by_id, edges, stats\n\n\nDEEPCENTER_VETO_DETECTOR = None # explicitly loaded and timed by diagnostic driver\n\ndef write_test_submission(tag: str = "base") -> None:\n    \n    \n    geffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))\n    print(f"Found {len(geffs)} prediction graphs")\n    if len(geffs) != len(test_stems):\n        found = {path.stem for path in geffs}\n        missing = sorted(set(test_stems) - found)\n        raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")\n\n    stats_rows: list[dict[str, object]] = []\n    seen_datasets: set[str] = set()\n    row_id = 0\n    total_nodes = 0\n    total_edges = 0\n\n    with SUBMISSION_PATH.open("w", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)\n        writer.writeheader()\n\n        for geff_path in geffs:\n            dataset = geff_path.stem\n            seen_datasets.add(dataset)\n            graph = graph_from_geff(geff_path)\n\n            nodes_by_id: dict[int, dict[str, object]] = {}\n            for row in graph.node_attrs().iter_rows(named=True):\n                node_id = int(row["node_id"])\n                nodes_by_id[node_id] = {\n                    "node_id": node_id,\n                    "t": int(row["t"]),\n                    "z": float(row["z"]),\n                    "y": float(row["y"]),\n                    "x": float(row["x"]),\n                }\n\n            raw_edges: list[dict[str, object]] = []\n            for row in graph.edge_attrs().iter_rows(named=True):\n                edge_prob = row.get("edge_prob") if hasattr(row, "get") else None\n                raw_edges.append({\n                    "source_id": int(row["source_id"]),\n                    "target_id": int(row["target_id"]),\n                    "edge_prob": None if edge_prob is None else float(edge_prob),\n                })\n\n            raw_node_count = len(nodes_by_id)\n            nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset, deepcenter_bundle=DEEPCENTER_VETO_DETECTOR)\n            if not nodes_by_id:\n                raise AssertionError(f"{dataset}: post-processing removed every node")\n\n            for node_id in sorted(nodes_by_id):\n                node = nodes_by_id[node_id]\n                writer.writerow({\n                    "id": row_id,\n                    "dataset": dataset,\n                    "row_type": "node",\n                    "node_id": int(node["node_id"]),\n                    "t": int(node["t"]),\n                    "z": max(0, int(round(float(node["z"])))),\n                    "y": max(0, int(round(float(node["y"])))),\n                    "x": max(0, int(round(float(node["x"])))),\n                    "source_id": -1,\n                    "target_id": -1,\n                })\n                row_id += 1\n\n            division_sources: dict[int, int] = {}\n            for edge in edges:\n                source_id = int(edge["source_id"])\n                target_id = int(edge["target_id"])\n                if source_id not in nodes_by_id or target_id not in nodes_by_id:\n                    raise AssertionError(f"{dataset}: dangling edge after filtering")\n                writer.writerow({\n                    "id": row_id,\n                    "dataset": dataset,\n                    "row_type": "edge",\n                    "node_id": -1,\n                    "t": -1,\n                    "z": -1,\n                    "y": -1,\n                    "x": -1,\n                    "source_id": source_id,\n                    "target_id": target_id,\n                })\n                row_id += 1\n                division_sources[source_id] = division_sources.get(source_id, 0) + 1\n\n            node_count = len(nodes_by_id)\n            edge_count = len(edges)\n            total_nodes += node_count\n            total_edges += edge_count\n            stats_rows.append({\n                "dataset": dataset,\n                "raw_nodes": raw_node_count,\n                "nodes": node_count,\n                "raw_edges": filter_stats["raw_edges"],\n                "edges": edge_count,\n                "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),\n                "edge_to_node_ratio": edge_count / max(node_count, 1),\n                "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),\n                **filter_stats,\n            })\n\n    expected_datasets = set(test_stems)\n    missing_datasets = sorted(expected_datasets - seen_datasets)\n    extra_datasets = sorted(seen_datasets - expected_datasets)\n    if missing_datasets or extra_datasets:\n        raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})\n    assert row_id == total_nodes + total_edges, "Internal row counter mismatch"\n    assert total_nodes > 0, "No node rows produced"\n\n    header = SUBMISSION_PATH.open().readline().strip().split(",")\n    assert header == CSV_COLUMNS, f"Bad CSV header: {header}"\n\n    stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)\n    stats["predict_minutes_total"] = predict_seconds / 60.0\n    stats["experiment_tag"] = f"{EXPERIMENT_TAG}:{tag}"\n    stats.to_csv(RUN_STATS_PATH, index=False)\n\n    print(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")\n    print(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")\n    print(f"Wrote {RUN_STATS_PATH}")\n    display(pd.read_csv(SUBMISSION_PATH, nrows=8))\n\n\nimport numpy as np\nfrom scipy.special import expit\nSCALE = np.array([1.625,0.40625,0.40625])\n\ndef point(row):\n    return np.array([row["z"], row["y"], row["x"]], dtype=float) * SCALE\n\ndef pair_features(coords):\n    """Physical-coordinate input [..., parent/d1/d2/g1/g2, z/y/x]."""\n    a = np.asarray(coords, dtype=float)\n    if a.shape[-2:] != (5, 3) or not np.isfinite(a).all():\n        raise ValueError("INVALID_PAIR_COORDINATES")\n    p, c1, c2, g1, g2 = np.moveaxis(a, -2, 0)\n    v1, v2 = c1-p, c2-p\n    norm = lambda v: np.linalg.norm(v, axis=-1)\n    d1, d2 = norm(v1), norm(v2)\n    sister, grand = norm(c1-c2), norm(g1-g2)\n    speed1, speed2 = norm(g1-c1), norm(g2-c2)\n    return np.stack([np.minimum(d1,d2), np.maximum(d1,d2), sister,\n                     norm((c1+c2)*0.5-p), np.abs(d1-d2)/np.maximum((d1+d2)*0.5,1e-6),\n                     np.sum(v1*v2,axis=-1)/np.maximum(d1*d2,1e-6), grand, grand-sister,\n                     (speed1+speed2)*0.5, np.abs(speed1-speed2)], axis=-1)\n\ndef context(nodes, out, parent, c1, c2):\n    t = int(nodes[parent]["t"])\n    if c1 == c2 or any(int(nodes[c]["t"]) != t+1 for c in (c1,c2)):\n        return None\n    if any(len(out.get(c,[])) != 1 for c in (c1,c2)):\n        return None\n    g1,g2 = out[c1][0],out[c2][0]\n    if g1 == g2 or any(int(nodes[g]["t"]) != t+2 for g in (g1,g2)):\n        return None\n    return np.stack([point(nodes[i]) for i in (parent,c1,c2,g1,g2)])\n\ndef design(x, mean, scale):\n    z = np.clip((np.asarray(x)-mean)/scale,-8.0,8.0)\n    return np.concatenate([np.ones((*z.shape[:-1],1)), z, z*z],axis=-1)\n\ndef predict(model, features):\n    return expit(design(features,np.asarray(model["mean"]),np.asarray(model["scale"]))@np.asarray(model["coefficients"]))\n\n"""Minimal additions to the original Forge proposal acceptance and ordering.\nNo fit, model update, or new eligibility rule. Explicit model selection.\n"""\nimport math\n\nclass ProposalPolicy:\n    def __init__(self, arm, score_pair):\n        if arm not in (\'A0\',\'G1\',\'R1\'): raise ValueError(arm)\n        self.arm=arm; self.score_pair=score_pair; self.rows=[]; self.frame_fallbacks=[]\n        self.scores={}\n    def admit(self, snapshot, proposal, dataset, t, existing_child):\n        distance,parent,child,*_=proposal\n        score=None if self.arm==\'A0\' else self.score_pair(snapshot,parent,existing_child,child,dataset)\n        if score is not None and not math.isfinite(score): raise RuntimeError(\'NONFINITE_LEARNED_SCORE\')\n        accept=not(self.arm==\'G1\' and score is not None and score<0.95)\n        row={\'dataset\':dataset,\'frame\':t,\'parent\':parent,\'child1\':existing_child,\'child2\':child,\n             \'arm\':self.arm,\'original_eligible\':True,\'learned_score\':score,\'abstain\':score is None and self.arm!=\'A0\',\n             \'accepted_before_sort\':accept,\'distance_priority\':distance,\'selected\':False}\n        self.rows.append(row);self.scores[(dataset,t,parent,child)]=score\n        return accept\n    def order(self, proposals, dataset, t):\n        before=[(x[1],x[2]) for x in proposals]\n        if self.arm==\'R1\':\n            scores=[self.scores[(dataset,t,p[1],p[2])] for p in proposals]\n            if any(v is None for v in scores):\n                self.frame_fallbacks.append((dataset,t)); proposals.sort(key=lambda p:p[0])\n            else:\n                proposals.sort(key=lambda p:(-self.scores[(dataset,t,p[1],p[2])],p[0]))\n        else:proposals.sort(key=lambda p:p[0])\n        assert set(before)=={(x[1],x[2]) for x in proposals}\n        for rank,p in enumerate(proposals):\n            row=next(r for r in reversed(self.rows) if (r[\'dataset\'],r[\'frame\'],r[\'parent\'],r[\'child2\'])==(dataset,t,p[1],p[2]))\n            row[\'original_order\']=before.index((p[1],p[2]));row[\'rank\']=rank\n    def selected(self,dataset,t,parent,child):\n        row=next(r for r in reversed(self.rows) if (r[\'dataset\'],r[\'frame\'],r[\'parent\'],r[\'child2\'])==(dataset,t,parent,child));row[\'selected\']=True\n\ndef patch_safe_div(source):\n    """Patch one audited pure function; caller never passes a training cell."""\n    append=\'                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\'\n    assert source.count(append)==1\n    source=source.replace(append,\'\'\'                proposal = (score, source_id, candidate_id, parent_dist, sister_dist)\n                if SPRINT_POLICY.admit((nodes_by_id, out_by_source), proposal, dataset, t, existing_child_id):\n                    proposals.append(proposal)\'\'\')\n    old=\'        proposals.sort(key=lambda item: item[0])\';assert source.count(old)==1\n    source=source.replace(old,\'        SPRINT_POLICY.order(proposals, dataset, t)\')\n    old=\'            used_targets.add(candidate_id)\';assert source.count(old)==1\n    return source.replace(old,\'            SPRINT_POLICY.selected(dataset, t, source_id, candidate_id)\\n\'+old)\n\nSPRINT_ARM=\'G1\'\nVALIDATION_EMBRYO_MAP={\'44b6_12dfb391\': \'44b6\', \'44b6_267148e4\': \'44b6\', \'44b6_2a2eff9f\': \'44b6\', \'44b6_341df25f\': \'44b6\', \'6bba_062c8d37\': \'6bba\', \'6bba_07e24132\': \'6bba\', \'6bba_085bf656\': \'6bba\', \'6bba_09961292\': \'6bba\'}\nSPRINT_PP_KEYS=[\'SAFE_DIV_MAX_UM\', \'SAFE_DIV_SISTER_MAX_UM\', \'SAFE_DIV_DIVERGE_UM\', \'SAFE_DIV_SISTER_SYMMETRY_TAU\', \'SAFE_DIV_EXISTING_CHILD_MAX_UM\', \'SAFE_DIV_FRAME_FRAC_CAP\', \'SAFE_DIV_GLOBAL_FRAC_CAP\', \'DEEPCENTER_SAFE_DIV_THRESHOLD\', \'DEEPCENTER_GAP_THRESHOLD\', \'GAP_CLOSE_UM\', \'OUTPUT_MIN_TRACK_LEN\', \'SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB\', \'MOTION_RELINK_TIGHT_UM\', \'MOTION_RELINK_RELAXED_UM\', \'GAP2_MAX_STEP_UM\', \'GAP2_MAX_TOTAL_UM\', \'MOTION_RELINK_LEARNED_BONUS\', \'MOTION_RELINK_VELOCITY_WEIGHT\', \'GAP_CLOSE_REUSE_UM\', \'OUTPUT_EDGE_MAX_UM\']\n"""Frozen-model inference and ordinary runtime evidence; inserted into Forge cell 5.\nSPRINT_ARM, VALIDATION_EMBRYO_MAP and original/patched functions supplied by builder.\n"""\nimport copy as _sprint_copy\nimport hashlib as _sprint_hashlib\nimport json as _sprint_json\n_SPRINT_WEIGHT_SHA=\'0a1f9b93bb529e70f4f7c2ba0907eea8b4cecd2befccc8ba1fb75e569edf77a0\'\n_sprint_paths=[p for p in GATE_ROOT.rglob(\'division_gate_weights.json\') if _sprint_hashlib.sha256(p.read_bytes()).hexdigest()==_SPRINT_WEIGHT_SHA]\nassert len(_sprint_paths)==1,\'SPRINT_EXACT_WEIGHT_REQUIRED\'\n_SPRINT_WEIGHTS=_sprint_json.loads(_sprint_paths[0].read_text())\nassert set(_SPRINT_WEIGHTS[\'held_out\'])=={\'44b6\',\'6bba\'}\n_SPRINT_TRAIN_RECEIPT=_sprint_json.loads((_sprint_paths[0].parent/\'division_training_receipt.json\').read_text())\nassert _SPRINT_TRAIN_RECEIPT[\'weights_sha256\']==_SPRINT_WEIGHT_SHA\nfor _fold in _SPRINT_TRAIN_RECEIPT[\'folds\']:assert _fold[\'held_out_embryo\'] not in _fold[\'training_embryos\']\nSPRINT_CALLS=[]\ndef _sprint_model_key(dataset):\n # Original validator switches the actual source directory; embryo comes only from the explicit frozen map.\n if Path(TEST_DIR).resolve()==(Path(COMP_DIR)/\'train\').resolve():\n  assert dataset in VALIDATION_EMBRYO_MAP,\'UNMAPPED_VALIDATION_EMBRYO\'\n  return VALIDATION_EMBRYO_MAP[dataset]\n return \'final\'\ndef _sprint_score(snapshot,parent,c1,c2,dataset):\n nodes,out_edges=snapshot;out={u:[int(e[\'target_id\']) for e in es] for u,es in out_edges.items() if u in (c1,c2)}\n coords=context(nodes,out,parent,c1,c2)\n if coords is None:return None\n key=_sprint_model_key(dataset);model=_SPRINT_WEIGHTS[\'final\'] if key==\'final\' else _SPRINT_WEIGHTS[\'held_out\'][key]\n return float(predict(model,pair_features(coords)))\ndef _sprint_graph_hash(nodes,edges):\n return _sprint_hashlib.sha256(_sprint_json.dumps([list(nodes.items()),edges],allow_nan=False,separators=(\',\',\':\')).encode()).hexdigest()\ndef sprint_audited_safe_div(nodes,edges,stats,**kw):\n global SPRINT_POLICY\n dataset=kw[\'dataset\'];SPRINT_POLICY=ProposalPolicy(SPRINT_ARM,_sprint_score)\n before=_sprint_graph_hash(nodes,edges)\n shadow_nodes=_sprint_copy.deepcopy(nodes);shadow_edges=_sprint_copy.deepcopy(edges);shadow_stats=dict(stats)\n result=_sprint_patched_safe_div(nodes,edges,stats,**kw)\n # Same snapshot original rule replay is evidence only; it never supplies output.\n old=_sprint_original_safe_div(shadow_nodes,shadow_edges,shadow_stats,**kw)\n assert _sprint_graph_hash(shadow_nodes,shadow_edges)==before,\'SPRINT_SHADOW_INPUT_MUTATED\'\n old_es={(int(e[\'source_id\']),int(e[\'target_id\'])) for e in old};new_es={(int(e[\'source_id\']),int(e[\'target_id\'])) for e in result}\n record={\'dataset\':dataset,\'arm\':SPRINT_ARM,\'model\':_sprint_model_key(dataset),\'input_hash\':before,\'old_safe_hash\':_sprint_graph_hash(shadow_nodes,old),\'new_safe_hash\':_sprint_graph_hash(nodes,result),\'added_edges\':sorted(new_es-old_es),\'lost_edges\':sorted(old_es-new_es),\'classifier_calls\':sum(r[\'learned_score\'] is not None for r in SPRINT_POLICY.rows),\'filtered\':sum(not r[\'accepted_before_sort\'] for r in SPRINT_POLICY.rows),\'abstain\':sum(r[\'abstain\'] for r in SPRINT_POLICY.rows),\'fallback_frames\':SPRINT_POLICY.frame_fallbacks,\'candidates\':SPRINT_POLICY.rows,\'resolved_config\':{k:globals()[k] for k in SPRINT_PP_KEYS}}\n SPRINT_CALLS.append(record)\n with (WORKING_DIR/\'sprint_production_calls.jsonl\').open(\'a\') as f:f.write(_sprint_json.dumps(record,allow_nan=False)+\'\\n\')\n print(\'SPRINT_PRODUCTION_SAFE_DIV\',dataset,record[\'model\'],record[\'classifier_calls\'],len(record[\'added_edges\']),len(record[\'lost_edges\']),flush=True)\n return result\nadd_safe_divisions_postlink=sprint_audited_safe_div\n\n\n', 'source/predict_unet_transformer.py': '#!/usr/bin/env python\n"""Run UNet + transformer edge prediction on datasets and export to .geff.\n\nUsage:\n    uv run scripts/predict_unet_transformer.py --split 0\n"""\n\nimport argparse\nimport contextlib\nimport json\nimport os\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport polars as pl\nimport torch\nimport torch.nn.functional as F\nimport zarr\nfrom tqdm import tqdm\n\nimport tracksdata as td\n\nfrom biohub_tracking.io import open_dataset, save_graph\n\n# Import model and helpers from companion training script.\nsys.path.insert(0, str(Path(__file__).parent))\nfrom train_unet_transformer import (\n    DEFAULT_METHOD,\n    UNetNodeTransformer,\n    extract_pos_features,\n    _POS_EMBED_DIM,\n)\nfrom biohub_tracking.models import TemporalUNet3D\n\nfrom dataspec import USERNAME, INTERACTIVE, WEIGHTS_PATH\nfrom evaluate import evaluate_run\nfrom biohub_tracking.metrics import summarise\n\n\n# =============================================================================\n# Prediction config\n# =============================================================================\n\n@dataclass\nclass PredictConfig:\n    """All hyperparameters that can affect prediction quality / score.\n\n    Detection\n    ---------\n    det_threshold : float\n        Minimum sigmoid probability for a local-max peak to be kept.\n\n    Edge filtering\n    --------------\n    edge_activation : str\n        Activation applied to raw edge logits: ``"sigmoid"`` (independent\n        per-edge scores) or ``"softmax"`` (row-normalised over t+1 nodes).\n    threshold : float\n        Minimum edge probability to consider a link at all.\n    max_parents_per_node : int\n        Maximum number of incoming edges per node (typically 1).\n    max_children_per_node : int\n        Maximum number of outgoing edges per node (1 = no divisions, 2 = divisions allowed).\n    """\n    # Detection\n    det_threshold: float = 0.5\n    det_tta: bool = True  # flip-xy TTA for detection logits\n    pool_kernel_um: float = 3.0  # max-pool kernel size in µm for detection peak extraction\n    # Edge filtering\n    edge_activation: str = "softmax"  # "sigmoid" or "softmax"\n    threshold: float = 0.5\n\n    # ILP post-processing\n    use_ilp: bool = False\n    ilp_edge_weight: float = -1.0\n    ilp_appearance_weight: float = 0.1\n    ilp_disappearance_weight: float = 0.1\n    ilp_division_weight: float = 1.0\n\n    max_parents_per_node: int | None = None\n    max_children_per_node: int | None = None\n\n    def __post_init__(self) -> None:\n        # When ILP is enabled it handles parent/children constraints itself,\n        # so greedy limits are left unconstrained (None).  When ILP is\n        # disabled, default to 1/1 to avoid unconstrained edge assignment.\n        if not self.use_ilp:\n            if self.max_parents_per_node is None:\n                self.max_parents_per_node = 1\n            if self.max_children_per_node is None:\n                self.max_children_per_node = 2\n\n\n\n# =============================================================================\n# Helpers\n# =============================================================================\n\n\n@contextlib.contextmanager\ndef suppress_output():\n    """Context manager to suppress stdout and stderr."""\n    with open(os.devnull, "w") as devnull:\n        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):\n            yield\n\n\n# =============================================================================\n# Graph building\n# =============================================================================\n\ndef build_graph(\n    coords: np.ndarray,\n    edges: list[tuple[int, int, float, float]],\n) -> td.graph.InMemoryGraph:\n    """Build a tracksdata graph from detection coords and predicted edges.\n\n    Avoids ``add_node_attr_key`` to sidestep a tracksdata/Polars compatibility\n    issue where the float default value is mistakenly used as a dtype.\n    Probabilities are passed as-is (softmax output, already in [0, 1]).\n    """\n    graph = td.graph.InMemoryGraph()\n\n    for key in ["z", "y", "x"]:\n        graph.add_node_attr_key(key, pl.Float64, -999999.0)\n\n    node_ids = graph.bulk_add_nodes([\n        {"t": int(t), "z": float(z), "y": float(y), "x": float(x)}\n        for t, z, y, x in coords\n    ])\n\n    if edges:\n        graph.add_edge_attr_key("edge_prob", pl.Float64, 0.0)\n        graph.add_edge_attr_key("edge_dist", pl.Float64, 0.0)\n        graph.bulk_add_edges([\n            {\n                "source_id": node_ids[src],\n                "target_id": node_ids[tgt],\n                "edge_prob": prob,\n                "edge_dist": dist,\n            }\n            for src, tgt, prob, dist in edges\n        ])\n\n    return graph\n\n\n# =============================================================================\n# Model loading\n# =============================================================================\n\n_DEFAULT_CONFIG = {\n    "unet_out_channels": 32,\n    "unet_layers": [32, 64, 128],\n    "downsample": [1, 4, 4],\n    "window_size": 2,\n}\n\n\ndef load_model(\n    weights_path: Path, device: torch.device,\n) -> tuple[UNetNodeTransformer, int, tuple[int, ...]]:\n    """Reconstruct UNetNodeTransformer from saved config + weights.\n\n    Reads ``config.json`` from the same directory as the weights file.\n    Falls back to ``_DEFAULT_CONFIG`` if the file is missing.\n\n    Returns ``(model, window_size, downsample)``.\n    """\n    config_path = weights_path.parent / "config.json"\n    if config_path.exists():\n        config = {**_DEFAULT_CONFIG, **json.loads(config_path.read_text())}\n    else:\n        print(f"Warning: config.json not found at {config_path}, using defaults.", flush=True)\n        config = _DEFAULT_CONFIG\n\n    # Support legacy configs that used "downsample_factor" (scalar).\n    if "downsample_factor" in config and "downsample" not in config:\n        df = config["downsample_factor"]\n        config["downsample"] = [df, df, df]\n\n    downsample = tuple(config["downsample"])\n\n    unet = TemporalUNet3D(\n        in_channels=1,\n        out_channels=config["unet_out_channels"],\n        layers=config["unet_layers"],\n    )\n    model = UNetNodeTransformer(\n        unet=unet,\n        unet_out_channels=config["unet_out_channels"],\n        pos_feat_dim=4 * _POS_EMBED_DIM,\n    )\n    state = torch.load(weights_path, map_location=device, weights_only=True)\n    model.load_state_dict(state)\n    model.to(device)\n    model.eval()\n    return model, config["window_size"], downsample\n\n\n# =============================================================================\n# Per-frame loading\n# =============================================================================\n\ndef _load_frame(\n    zarr_arr,\n    t: int,\n    target_shape: list[int],\n    downsample: tuple[int, ...] = (1, 1, 1),\n) -> torch.Tensor:\n    """Load one frame from zarr with strided spatial downsample (no normalisation)."""\n    dz, dy, dx = downsample\n    raw = zarr_arr[t, ::dz, ::dy, ::dx].astype(np.float32)\n    frame = torch.from_numpy(raw)\n    if list(frame.shape) != target_shape:\n        frame = F.interpolate(\n            frame[None, None], size=target_shape,\n            mode="trilinear", align_corners=False,\n        )[0, 0]\n    return frame\n\n\n# =============================================================================\n# Inference\n# =============================================================================\n\ndef pool_kernel_from_um(\n    um: float,\n    voxel_size: tuple[float, ...],\n) -> tuple[int, ...]:\n    """Convert a physical suppression distance (microns) to a per-axis voxel kernel.\n\n    Each axis gets ``round(um / voxel_size_axis)`` voxels, forced to odd\n    (for symmetric padding) and at least 1.\n\n    Parameters\n    ----------\n    um : float\n        Desired suppression distance in microns.\n    voxel_size : tuple[float, ...]\n        Per-axis voxel sizes in microns, e.g. ``(1.625, 0.40625, 0.40625)``.\n    """\n    kernel = []\n    for s in voxel_size:\n        k = max(1, round(um / s))\n        if k % 2 == 0:\n            k += 1\n        kernel.append(k)\n    return tuple(kernel)\n\n\ndef _detect_cells_pooled(\n    det_logits: torch.Tensor,\n    t: int,\n    det_threshold: float = 0.5,\n    pool_kernel: tuple[int, ...] = (3, 3, 3),\n) -> np.ndarray:\n    """Extract cell coordinates via max-pool local-max (same as training).\n\n    Coordinates are returned in the downsampled grid.  The caller is\n    responsible for scaling back to original resolution if needed.\n\n    Parameters\n    ----------\n    det_logits : torch.Tensor\n        (1, Z, Y, X) raw logits.\n    t : int\n        Time index to prepend as the first column.\n    det_threshold : float\n        Minimum sigmoid probability for a peak to be considered (default 0.5).\n    pool_kernel : tuple[int, ...]\n        Per-axis kernel size for local-max pooling,\n        e.g. ``(3, 11, 11)`` for anisotropic data.\n\n    Returns\n    -------\n    np.ndarray\n        (N, 4) int16 array with columns [t, z, y, x] in downsampled space.\n    """\n    logits = det_logits.unsqueeze(0)  # (1, 1, Z, Y, X)\n    pad = tuple(k // 2 for k in pool_kernel)\n    pooled = F.max_pool3d(logits, pool_kernel, stride=1, padding=pad)\n    is_peak = (logits == pooled) & (torch.sigmoid(logits) > det_threshold)\n    peak_idx = torch.nonzero(is_peak[0, 0])  # (N, 3)\n\n    if peak_idx.shape[0] == 0:\n        return np.empty((0, 4), dtype=np.int16)\n\n    coords = peak_idx.float().cpu().numpy()\n    t_col = np.full((len(coords), 1), t, dtype=np.float32)\n    return np.concatenate([t_col, coords], axis=1).astype(np.int16)\n\n\n@torch.no_grad()\ndef predict_video(\n    model: UNetNodeTransformer,\n    ds_path: Path,\n    device: torch.device,\n    cfg: PredictConfig,\n    window_size: int = 2,\n    max_frames: int | None = None,\n    unet_batch_size: int = 4,\n    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:\n    """Run inference on a single video using sliding windows of W frames.\n\n    Windows slide with stride ``W - 1`` so every consecutive pair is covered\n    exactly once.  UNet features from each window are reused for edge\n    prediction on all ``W - 1`` consecutive pairs within the window.\n\n    Returns\n    -------\n    coords : np.ndarray\n        Shape (N, 4) — columns [t, z, y, x] in original resolution.\n    edges : list of (src_idx, tgt_idx, prob, distance) tuples\n    """\n    ds = open_dataset(ds_path, normalize=False, load_image=False, downsample=downsample)\n    if "0.001" not in ds.quantiles or "0.999" not in ds.quantiles:\n        raise ValueError(f"Zarr attrs missing image_statistics.quantiles for {ds_path}")\n    zarr_arr = zarr.open_group(str(ds.zarr_path), mode="r")["0"]\n    q_low = float(ds.quantiles["0.001"])\n    q_high = float(ds.quantiles["0.999"])\n\n    T = ds.image_shape[0] if max_frames is None else min(ds.image_shape[0], max_frames)\n    image_shape = (T,) + ds.image_shape[1:]\n    target_shape = list(image_shape[1:])\n\n    ds_arr = np.array(downsample, dtype=np.float32)  # for coord rescaling at the end\n    ds_arr_t = torch.from_numpy(ds_arr).to(device)   # for predict_edges (original-space coords)\n    pos_feat_dim = 4 * _POS_EMBED_DIM\n    W = window_size\n    voxel_size = tuple(s * d for s, d in zip(ds.scale, downsample))\n    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n\n    # Running node registry — each entry records the frame-t detections.\n    # coord_offset[t] = (start, end) half-open range into the stacked array.\n    seen_frames: set[int] = set()\n    seen_pairs: set[tuple[int, int]] = set()\n    coord_lists: list[np.ndarray] = []\n    coord_offset: dict[int, tuple[int, int]] = {}\n    global_node_count: int = 0\n    all_edges: list[tuple[int, int, float, float]] = []\n\n    # Sliding windows with stride W-1 cover every consecutive pair exactly once.\n    stride = max(W - 1, 1)\n    window_starts = list(range(0, T - W + 1, stride))\n    # Ensure the very last pair (T-2 → T-1) is covered.\n    if not window_starts or window_starts[-1] + W < T:\n        last = max(T - W, 0)\n        if not window_starts or last != window_starts[-1]:\n            window_starts.append(last)\n\n    for ws in tqdm(\n        window_starts,\n        desc="  windows",\n        leave=False,\n        disable=not INTERACTIVE,\n    ):\n        frame_indices = list(range(ws, ws + W))\n\n        # --- UNet encode (single window, batch_size=1) ---\n        imgs = torch.stack([\n            _load_frame(zarr_arr, t, target_shape, downsample)\n            for t in frame_indices\n        ])  # (W, *spatial)\n        # Quantile normalisation (0.1%–99.9%) to match training pipeline.\n        imgs = ((imgs - q_low) / (q_high - q_low + 1e-6)).clamp(0.0)\n        imgs = imgs.unsqueeze(0).to(device)   # (1, W, *spatial)\n\n        unet_out, det_logits = model.encode(imgs)\n        # unet_out: (1, W, C, *spatial_down), det_logits: list of W × (1, 1, *spatial_down)\n\n        # Detection TTA: original + flip-x + flip-y + flip-xy, average logits.\n        # TTA: flip along Y (-2) and X (-1) only.  Z is excluded because\n        # the data is highly anisotropic (Z resolution ~4x coarser than XY),\n        # so Z-flips would produce out-of-distribution inputs.\n        if cfg.det_tta:\n            _edge_tta = os.environ.get(\'BIOHUB_EDGE_FEATURE_TTA\', \'0\') != \'0\'\n            _unet_acc = unet_out.clone() if _edge_tta else None\n            _nv = 1\n            for dims in [(-1,), (-2,), (-2, -1)]:\n                imgs_flip = imgs.flip(dims)\n                _u_flip, det_flip = model.encode(imgs_flip)\n                for f in range(W):\n                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)\n                if _edge_tta:\n                    _unet_acc = _unet_acc + _u_flip.flip(dims)\n                del imgs_flip, det_flip, _u_flip\n                _nv += 1\n            for _k in (1, 3):\n                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                _u_rot, det_rot = model.encode(imgs_rot)\n                for f in range(W):\n                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))\n                if _edge_tta:\n                    _unet_acc = _unet_acc + torch.rot90(_u_rot, -_k, dims=(-2, -1))\n                del imgs_rot, det_rot, _u_rot\n                _nv += 1\n            imgs_t = imgs.transpose(-1, -2)\n            _u_t, det_t = model.encode(imgs_t)\n            for f in range(W):\n                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)\n            if _edge_tta:\n                _unet_acc = _unet_acc + _u_t.transpose(-1, -2)\n            del imgs_t, det_t, _u_t\n            _nv += 1\n            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)\n            _u_at, det_at = model.encode(imgs_at)\n            for f in range(W):\n                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))\n            if _edge_tta:\n                _unet_acc = _unet_acc + torch.rot90(_u_at.transpose(-1, -2), -1, dims=(-2, -1))\n            del imgs_at, det_at, _u_at\n            _nv += 1\n            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n            if _edge_tta:\n                if _unet_acc.shape != unet_out.shape:\n                    raise RuntimeError(\'EDGE-TTA SHAPE MISMATCH: %s vs %s\'\n                                       % (tuple(_unet_acc.shape), tuple(unet_out.shape)))\n                _delta = float((_unet_acc / _nv - unet_out).abs().mean())\n                if _delta == 0.0:\n                    raise RuntimeError(\'EDGE-TTA NO-OP: averaged features bit-identical to the \'\n                                       \'single-pass features, so the augmented encodes \'\n                                       \'contributed nothing and this arm would read as a \'\n                                       \'false null\')\n                unet_out = _unet_acc / _nv\n                print(\'EDGE_TTA_ACTIVE views=\', _nv, \'mean_abs_feat_delta=\', round(_delta, 6), flush=True)\n                del _unet_acc\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n            _secondary_edge_tta = os.environ.get(\n                "BIOHUB_SECONDARY_EDGE_FEATURE_TTA", "0"\n            ) != "0"\n            _secondary_unet_acc = (\n                secondary_unet_out.clone() if _secondary_edge_tta else None\n            )\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _secondary_u_flip, secondary_det_flip = secondary_model.encode(\n                            secondary_imgs_flip\n                        )\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        if _secondary_edge_tta:\n                            _secondary_unet_acc = _secondary_unet_acc + _secondary_u_flip.flip(dims)\n                        del secondary_imgs_flip, secondary_det_flip, _secondary_u_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _secondary_u_rot, secondary_det_rot = secondary_model.encode(\n                            secondary_imgs_rot\n                        )\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        if _secondary_edge_tta:\n                            _secondary_unet_acc = _secondary_unet_acc + torch.rot90(\n                                _secondary_u_rot, -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot, _secondary_u_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _secondary_u_t, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    if _secondary_edge_tta:\n                        _secondary_unet_acc = _secondary_unet_acc + _secondary_u_t.transpose(-1, -2)\n                    del secondary_imgs_t, secondary_det_t, _secondary_u_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _secondary_u_at, secondary_det_at = secondary_model.encode(\n                        secondary_imgs_at\n                    )\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    if _secondary_edge_tta:\n                        _secondary_unet_acc = _secondary_unet_acc + torch.rot90(\n                            _secondary_u_at.transpose(-1, -2), -1, dims=(-2, -1)\n                        )\n                    del secondary_imgs_at, secondary_det_at, _secondary_u_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n                    if _secondary_edge_tta:\n                        if _secondary_unet_acc.shape != secondary_unet_out.shape:\n                            raise RuntimeError("SECONDARY_EDGE_TTA_SHAPE_MISMATCH")\n                        _secondary_delta = float(\n                            (_secondary_unet_acc / _secondary_nv - secondary_unet_out).abs().mean()\n                        )\n                        if _secondary_delta == 0.0:\n                            raise RuntimeError("SECONDARY_EDGE_TTA_NO_OP")\n                        _secondary_edge_tta_weight = float(os.environ.get(\n                            "BIOHUB_SECONDARY_EDGE_FEATURE_TTA_WEIGHT", "1.0"\n                        ))\n                        if not 0.0 < _secondary_edge_tta_weight <= 1.0:\n                            raise RuntimeError("SECONDARY_EDGE_TTA_BAD_WEIGHT")\n                        _secondary_tta_mean = _secondary_unet_acc / _secondary_nv\n                        secondary_unet_out = (\n                            (1.0 - _secondary_edge_tta_weight) * secondary_unet_out\n                            + _secondary_edge_tta_weight * _secondary_tta_mean\n                        )\n                        print(\n                            "SECONDARY_EDGE_TTA_ACTIVE views=",\n                            _secondary_nv,\n                            "weight=",\n                            _secondary_edge_tta_weight,\n                            "mean_abs_feat_delta=",\n                            round(_secondary_delta, 6),\n                            flush=True,\n                        )\n                        del _secondary_unet_acc\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    blended_det = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n                    primary_candidates = len(_detect_cells_pooled(\n                        primary_det[0],\n                        int(frame_indices[f]),\n                        cfg.det_threshold,\n                        pool_k,\n                    ))\n                    blended_candidates = len(_detect_cells_pooled(\n                        blended_det[0],\n                        int(frame_indices[f]),\n                        cfg.det_threshold,\n                        pool_k,\n                    ))\n                    minimum_retention = float(os.environ.get(\n                        "BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION",\n                        "0.90",\n                    ))\n                    candidate_retention = (\n                        blended_candidates / primary_candidates\n                        if primary_candidates\n                        else 1.0\n                    )\n                    use_primary_detection = bool(\n                        primary_candidates > 0\n                        and candidate_retention < minimum_retention\n                    )\n                    det_logits[f] = (\n                        primary_det if use_primary_detection else blended_det\n                    )\n                    if int(frame_indices[f]) not in seen_frames:\n                        shard = os.environ.get(\n                            "BIOHUB_GPU_SHARD", "single"\n                        ).replace("/", "_")\n                        guard_log = (\n                            TRACE.root\n                            / f"retention_guard_{shard}.jsonl"\n                        )\n                        guard_record = {\n                            "dataset": ds_path.stem,\n                            "frame": int(frame_indices[f]),\n                            "primary_candidates": int(primary_candidates),\n                            "blended_candidates": int(blended_candidates),\n                            "retention": float(candidate_retention),\n                            "minimum_retention": float(minimum_retention),\n                            "secondary_detection_weight": float(secondary_detection_weight),\n                            "det_threshold": float(cfg.det_threshold),\n                            "use_primary": bool(use_primary_detection),\n                        }\n                        with guard_log.open("a") as guard_handle:\n                            guard_handle.write(\n                                json.dumps(guard_record, sort_keys=True)\n                                + "\\n"\n                            )\n                        if use_primary_detection:\n                            print(\n                                "BIOHUB_RETENTION_GUARD "\n                                + json.dumps(guard_record, sort_keys=True),\n                                flush=True,\n                            )\n\n            del secondary_det_logits\n\n        TRACE.window(frame_indices, unet_out, det_logits, secondary_unet_out)\n        del imgs\n\n        # --- Detect cells in each frame (dedup across windows) ---\n        for f_idx, t in enumerate(frame_indices):\n            if t not in seen_frames:\n                arr = _detect_cells_pooled(\n                    det_logits[f_idx][0], t, cfg.det_threshold, pool_k,\n                )\n                coord_offset[t] = (global_node_count, global_node_count + len(arr))\n                global_node_count += len(arr)\n                coord_lists.append(arr)\n                seen_frames.add(t)\n\n        coords_so_far = (\n            np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.int16)\n        )\n\n        # --- Edge prediction for each consecutive pair in the window ---\n        for f_idx in range(W - 1):\n            t_src, t_tgt = frame_indices[f_idx], frame_indices[f_idx + 1]\n            if (t_src, t_tgt) in seen_pairs:\n                continue\n            seen_pairs.add((t_src, t_tgt))\n\n            if t_src not in coord_offset or t_tgt not in coord_offset:\n                continue\n            s_src, e_src = coord_offset[t_src]\n            s_tgt, e_tgt = coord_offset[t_tgt]\n            if e_src == s_src or e_tgt == s_tgt:\n                continue\n\n            c_src = coords_so_far[s_src:e_src]\n            c_tgt = coords_so_far[s_tgt:e_tgt]\n            n_src, n_tgt = len(c_src), len(c_tgt)\n            idx_src = np.arange(s_src, e_src, dtype=np.int64)\n            idx_tgt = np.arange(s_tgt, e_tgt, dtype=np.int64)\n\n            # Build tensors (batch_size=1).\n            p_coords_src = torch.from_numpy(c_src[:, 1:].astype(np.float32)).unsqueeze(0).to(device)\n            p_coords_tgt = torch.from_numpy(c_tgt[:, 1:].astype(np.float32)).unsqueeze(0).to(device)\n            # Use window-relative time (f_idx, f_idx+1) normalised by W, not absolute frame index.\n            window_shape = (W,) + image_shape[1:]\n            c_src_rel = c_src.copy()\n            c_src_rel[:, 0] = f_idx\n            c_tgt_rel = c_tgt.copy()\n            c_tgt_rel[:, 0] = f_idx + 1\n            p_pos_src = torch.from_numpy(extract_pos_features(c_src_rel, window_shape)).unsqueeze(0).to(device)\n            p_pos_tgt = torch.from_numpy(extract_pos_features(c_tgt_rel, window_shape)).unsqueeze(0).to(device)\n            p_mask_src = torch.ones(1, n_src, dtype=torch.bool, device=device)\n            p_mask_tgt = torch.ones(1, n_tgt, dtype=torch.bool, device=device)\n\n            unet_feat_src = model._index_features(\n                unet_out[:, f_idx], p_coords_src, p_mask_src,\n            )\n            unet_feat_tgt = model._index_features(\n                unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n            )\n            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            _bidirectional_weight = float(\n                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")\n            )\n            if not globals().get("_trio_harmonic_recorded", False):\n                import json as _trio_json\n                from pathlib import Path as _TrioPath\n                _TrioPath("/kaggle/working/trio_harmonic_" + str(os.getpid()) + ".json").write_text(_trio_json.dumps({"weight": _bidirectional_weight, "mode": "harmonic_probability"}))\n                globals()["_trio_harmonic_recorded"] = True\n            if _bidirectional_weight > 0.0:\n                reverse_logits_native = model.predict_edges(\n                    unet_feat_tgt, unet_feat_src,\n                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,\n                    p_pos_tgt, p_pos_src,\n                    p_mask_tgt, p_mask_src,\n                )  # (1, n_tgt, n_src)\n                reverse_logits_pair = reverse_logits_native.transpose(1, 2)\n\n                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                forward_scale = edge_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)\n                reverse_scale = reverse_logits_pair.float().std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)\n                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)\n                reverse_aligned = (\n                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio\n                    + forward_center\n                )\n                # Biohub 145: require mutual forward/reverse support in probability space.\n                # The harmonic mean penalizes a candidate when either temporal direction\n                # assigns it very low probability, while calibration preserves the forward\n                # logit scale used by the unchanged downstream candidate threshold and ILP.\n                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)\n                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)\n                harmonic_prob = 1.0 / (\n                    (1.0 - _bidirectional_weight) / forward_prob\n                    + _bidirectional_weight / reverse_prob\n                )\n                harmonic_prob = harmonic_prob / harmonic_prob.sum(\n                    dim=1, keepdim=True\n                ).clamp_min(1e-8)\n                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))\n                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)\n                harmonic_scale = harmonic_logits.std(\n                    dim=1, keepdim=True, unbiased=False\n                ).clamp_min(1e-4)\n                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)\n                edge_logits_pair = (\n                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio\n                    + forward_center\n                ).to(reverse_aligned.dtype)\n                del (\n                    reverse_logits_native,\n                    reverse_logits_pair,\n                    reverse_aligned,\n                    forward_prob,\n                    reverse_prob,\n                    harmonic_prob,\n                    harmonic_logits,\n                )\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]\n            if cfg.edge_activation == "softmax":\n                probs = torch.softmax(raw, dim=0).cpu().numpy()\n            else:\n                probs = torch.sigmoid(raw).cpu().numpy()\n\n            TRACE.edges(frame_indices, idx_src, idx_tgt, coords_so_far, probs, cfg.threshold)\n            candidates = sorted(\n                [\n                    (probs[i, j], i, j)\n                    for i in range(n_src)\n                    for j in range(n_tgt)\n                    if probs[i, j] > cfg.threshold\n                ],\n                reverse=True,\n            )\n\n            children_count: dict[int, int] = {}\n            parents_count: dict[int, int] = {}\n\n            for prob, i, j in candidates:\n                n_ch = children_count.get(i, 0)\n                n_pa = parents_count.get(j, 0)\n                if cfg.max_children_per_node is not None and n_ch >= cfg.max_children_per_node:\n                    continue\n                if cfg.max_parents_per_node is not None and n_pa >= cfg.max_parents_per_node:\n                    continue\n\n                gi, gj = int(idx_src[i]), int(idx_tgt[j])\n                dist = float(np.linalg.norm(\n                    coords_so_far[gi, 1:].astype(np.float32)\n                    - coords_so_far[gj, 1:].astype(np.float32)\n                ))\n                all_edges.append((gi, gj, float(prob), dist))\n                children_count[i] = n_ch + 1\n                parents_count[j] = n_pa + 1\n\n        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n\n    coords = np.concatenate(coord_lists) if coord_lists else np.empty((0, 4), dtype=np.int16)\n    # Scale spatial coords back to original resolution.\n    coords = coords.astype(np.float32)\n    coords[:, 1:] *= ds_arr\n    coords = coords.astype(np.int16)\n\n    # Label-free, pre-ILP detector-coordinate manifest. This executes inside\n    # predict_video, before build_graph and the ILP call in predict().\n    _coordinate_manifest_arm = os.environ.get(\n        "BIOHUB_DIAGNOSTIC_ARM", ""\n    ).strip()\n    if _coordinate_manifest_arm:\n        import hashlib as _coordinate_hashlib\n\n        _coordinate_shard = os.environ.get(\n            "BIOHUB_GPU_SHARD", "single"\n        ).replace("/", "_")\n        _coordinate_array = np.ascontiguousarray(\n            coords.astype("<i2", copy=False)\n        )\n        _coordinate_frame_counts = [\n            [int(_coordinate_t), int((_coordinate_array[:, 0] == _coordinate_t).sum())]\n            for _coordinate_t in np.unique(_coordinate_array[:, 0])\n        ]\n        _coordinate_record = {\n            "columns": ["t", "z", "y", "x"],\n            "coordinate_sha256": _coordinate_hashlib.sha256(\n                _coordinate_array.tobytes(order="C")\n            ).hexdigest(),\n            "dataset": ds_path.stem,\n            "dtype": "<i2",\n            "frame_counts": _coordinate_frame_counts,\n            "rows": int(len(_coordinate_array)),\n            "stage": "post_detection_pre_graph_pre_ilp",\n        }\n        _coordinate_manifest_path = (\n            TRACE.root\n            / f"detector_coordinates_{_coordinate_manifest_arm}_"\n            f"{_coordinate_shard}.jsonl"\n        )\n        with _coordinate_manifest_path.open("a") as _coordinate_handle:\n            _coordinate_handle.write(\n                json.dumps(_coordinate_record, sort_keys=True) + "\\n"\n            )\n\n    return coords, all_edges\n\n\n# =============================================================================\n# Prediction loop\n# =============================================================================\n\ndef predict(\n    data_dir: Path,\n    fold: int,\n    splits_file: Path,\n    weights_path: Path,\n    cfg: PredictConfig,\n    method: str = DEFAULT_METHOD,\n    debug_video: Path | None = None,\n    unet_batch_size: int = 4,\n    video_slice: slice | None = None,\n    evaluate: bool = False,\n) -> None:\n    """Run inference on the test split and save predictions as .geff files."""\n    if debug_video is not None:\n        test_names = [debug_video.name]\n        data_dir = debug_video.parent\n    else:\n        folds = json.loads(splits_file.read_text())\n        test_names = folds[fold]["test"]\n        if video_slice is not None:\n            test_names = test_names[video_slice]\n\n    from dataspec import PREDICTIONS_PATH\n    output_dir = PREDICTIONS_PATH / USERNAME / method / f"split_{fold}"\n    if output_dir.exists():\n        import shutil\n        for old in output_dir.glob("*.geff"):\n            if old.is_dir():\n                shutil.rmtree(old)\n            else:\n                old.unlink()\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print(\n        f"Fold {fold}: {len(test_names)} datasets | "\n        f"weights={weights_path} | device={device} | window_size={window_size} | pool_kernel_um={cfg.pool_kernel_um}",\n        flush=True,\n    )\n\n    for name in tqdm(test_names, desc="Predicting", disable=not INTERACTIVE):\n        ds_path = data_dir / name\n        coords, edges = predict_video(\n                model, ds_path, device,\n                cfg=cfg,\n                window_size=window_size,\n                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )\n        graph = build_graph(coords, edges)\n        if cfg.use_ilp and graph.num_edges() > 0:\n            solver = td.solvers.ILPSolver(\n                edge_weight=cfg.ilp_edge_weight * td.EdgeAttr("edge_prob"),\n                appearance_weight=cfg.ilp_appearance_weight,\n                disappearance_weight=cfg.ilp_disappearance_weight,\n                division_weight=cfg.ilp_division_weight,\n            )\n            with suppress_output():\n                graph = solver.solve(graph)\n        save_graph(graph, output_dir / f"{name}.geff")\n\n    print(f"Saved {len(test_names)} predictions to {output_dir}", flush=True)\n\n    if evaluate:\n        run = {\n            "username": USERNAME,\n            "method": method,\n            "split": f"split_{fold}",\n            "dir": output_dir,\n            "geffs": sorted(output_dir.glob("*.geff")),\n        }\n        results = evaluate_run(run)\n        s = summarise(results)\n        print(\n            f"Evaluation ({len(results)} videos): "\n            f"score={s[\'score\']:.4f}  "\n            f"edge_jaccard={s[\'edge_jaccard\']:.4f}  "\n            f"adj_edge_jaccard={s[\'adj_edge_jaccard\']:.4f} (n_adj={s[\'n_adj\']})  "\n            f"division_jaccard={s[\'division_jaccard\']:.4f} "\n            f"(TP={s[\'division_tp\']} FP={s[\'division_fp\']} FN={s[\'division_fn\']})  "\n            f"node_recall={s[\'node_recall\']:.4f}  (n={s[\'n\']})",\n            flush=True,\n        )\n\n\n# =============================================================================\n# CLI\n# =============================================================================\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(\n        description="Run UNet + transformer edge prediction.",\n        formatter_class=argparse.RawDescriptionHelpFormatter,\n    )\n    parser.add_argument("--method", type=str, default=DEFAULT_METHOD)\n    parser.add_argument("--data-dir", type=str, default=None,\n                        help="Default: DATASET_PATH")\n    parser.add_argument("--splits", type=str, default=None,\n                        help="Default: DATASET_PATH/dataset_splits.json")\n    parser.add_argument("--split", type=str, default="0",\n                        help="Split index (0-4) or \'all\'.")\n    parser.add_argument("--weights", type=str, default=None,\n                        help="Path to weights file. "\n                             "Default: weights/{method}/split_{split}/edge_predictor_best.pth")\n    parser.add_argument("--debug-video", type=str, default=None,\n                        help="Path to a single dataset. Ignores fold/splits.")\n    parser.add_argument("--slice", type=str, default=None,\n                        help="Python slice of the test list, e.g. \':1\' for first video, "\n                             "\'2:5\' for videos 2-4.")\n    parser.add_argument("--unet-batch-size", type=int, default=4,\n                        help="Number of frame pairs per UNet forward pass (default: 4).")\n    parser.add_argument("--evaluate", action="store_true",\n                        help="Run evaluation against GT after saving predictions.")\n    parser.add_argument("--det-threshold", type=float, default=0.99,\n                        help="Min sigmoid probability for a detection peak to be kept. "\n                             "Default 0.99: the detector is poorly calibrated because the "\n                             "ground truth is sparse (only some cells annotated), so a high "\n                             "threshold keeps precision up. Sweep it for your model.")\n    parser.add_argument("--use-ilp", action="store_true",\n                        help="Post-process the predicted graph with the tracksdata ILP "\n                             "solver (global, flow-consistent linking) instead of greedy "\n                             "assignment. Needs pyscipopt; produces cleaner tracks.")\n    parser.add_argument("--ilp-edge-weight", type=float, default=-1.0,\n                        help="ILP: weight on edge_prob (default -1.0).")\n    parser.add_argument("--ilp-appearance-weight", type=float, default=0.1,\n                        help="ILP: cost of a track appearing (default 0.1).")\n    parser.add_argument("--ilp-disappearance-weight", type=float, default=0.1,\n                        help="ILP: cost of a track disappearing (default 0.1).")\n    parser.add_argument("--ilp-division-weight", type=float, default=1.0,\n                        help="ILP: cost of a division; lower to allow more splits (default 1.0).")\n\n    args = parser.parse_args()\n\n    from dataspec import DATASET_PATH\n    data_dir = Path(args.data_dir) if args.data_dir else Path(DATASET_PATH)\n    splits_file = Path(args.splits) if args.splits else data_dir / "dataset_splits.json"\n    debug_video = Path(args.debug_video) if args.debug_video else None\n    video_slice = (\n        slice(*[int(x) if x else None for x in args.slice.split(":")])\n        if args.slice else None\n    )\n    cfg = PredictConfig(\n        det_threshold=args.det_threshold,\n        use_ilp=args.use_ilp,\n        ilp_edge_weight=args.ilp_edge_weight,\n        ilp_appearance_weight=args.ilp_appearance_weight,\n        ilp_disappearance_weight=args.ilp_disappearance_weight,\n        ilp_division_weight=args.ilp_division_weight,\n    )\n\n    folds = range(5) if args.split == "all" else [int(args.split)]\n\n    for fold in folds:\n        weights_path = (\n            Path(args.weights) if args.weights\n            else WEIGHTS_PATH / args.method / f"split_{fold}" / "edge_predictor_best.pth"\n        )\n        predict(\n            data_dir=data_dir,\n            fold=fold,\n            splits_file=splits_file,\n            weights_path=weights_path,\n            cfg=cfg,\n            method=args.method,\n            debug_video=debug_video,\n            unet_batch_size=args.unet_batch_size,\n            video_slice=video_slice,\n            evaluate=args.evaluate,\n        )\n\n\nif __name__ == "__main__":\n    main()\n'}
HASHES={'prepare_bundle.py': '3819b84aa9ecb4335a5cf84c0be315ec1ee95a029fdab8b42e775f0b3abd9785', 'convert_bundle.py': '43b24eac76e9af733221e872be35bc731b9c636fd944cc043280fc9c87155ff3', 'install_offline.py': '97e81e711ca199ebb656e9223a583fc7a6cc3e9d36276d98f4e3d9fc09be99ad', 'chain.py': 'df1b2c6af55ceb5ab0698ffc0e8879afcaf1fce0393c92b7b5fa8722cec5c591', 'diagnostic_validate.py': '3ddd63af9163897b840a81a8ff77560d4fb0e170b309b9a72b76b5e8d21f1cf1', 'trace_runtime.py': 'abeb6cdc13fc558c81e4afa4451d876b8f5a87da38f5d03c9f3398d87b8857c7', 'source_hashes.json': '30dc1139566780d88a3da5be45da938df0b7ef93373f9e5b4e30a31f633fdaa8', 'dependency_specs.json': '7dc29b6e4b0a5c7a11165ff7dfe8a6f5ddb962a6e1d08b4b5e548692b080c540', 'source/config_1.py': '9662daf08240b4a6b350bae3454947ee825419d8db67ddbb667147b036075293', 'source/config_2.py': 'b42b9653cb0b931d6cba3a5218e1799e78fd997c21f0dcf317514cf2838d0bea', 'source/config_3.py': '77fdadffbd02c87e8f686e55944b6741312be552f34c8dc615e6391a1959a2dc', 'source/official_reader.py': 'c0fe1868d4550235c4673d4a3fc5e2d6ab0402f3cd0bac73eec8d1348eb9690d', 'source/patch_support.py': 'c74501d67c8d6cb93ff1b81143fedfdee600ad0e02b9382c53da3025d062c403', 'source/postprocess.py': 'd25d3b410279dbd296852724747d39545c8a356d2180f9d32b607beaa9e633e3', 'source/predict_unet_transformer.py': 'f4a9ad4efca54454c4cb8f39e3a37c200dd12fdba37293e2a2b7bb531d4bdef6'}
SOURCE_DIGEST='ba66910c313b00755cf5328a8babca6c9e5b5a72195b1a545888ac44eccbf89d'
started={'task':TASK,'request_id':'CPU-BATCH-PREP-01','status':'STARTED','utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),'source_digest':SOURCE_DIGEST}
print('BATCH_STARTED',json.dumps(started),flush=True)
(ROOT/'batch_started.json').write_text(json.dumps(started,indent=2)+'\n')
for name,body in PAYLOAD.items():
    p=ROOT/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(body)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==HASHES[name],name
(ROOT/'submitted_source_hashes.json').write_text(json.dumps(HASHES,indent=2)+'\n')


In [ ]:
os.environ.update(CPU_BATCH_TASK_ID=TASK,CPU_PREP_DEADLINE_EPOCH=str(time.time()+1800),OMP_NUM_THREADS='2',MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',POLARS_MAX_THREADS='2',PYTHONUNBUFFERED='1')
p=subprocess.Popen([sys.executable,str(ROOT/'prepare_bundle.py')],start_new_session=True)
try:
    rc=p.wait(timeout=max(1,float(os.environ['CPU_PREP_DEADLINE_EPOCH'])-time.time()))
    if rc:raise RuntimeError(f'Preparation child exited {rc}')
except BaseException as e:
    if p.poll() is None:os.killpg(p.pid,signal.SIGKILL)
    record={'task':TASK,'status':'FAILED','error_type':type(e).__name__,'error':str(e),'traceback':traceback.format_exc()}
    (ROOT/'batch_parent_error.json').write_text(json.dumps(record,indent=2)+'\n');print('BATCH_FAILED',json.dumps(record),flush=True);raise
print('BATCH_PREPARATION_FINISHED',flush=True)
